# Step 5 word2vec の原理 — 分布仮説から word embedding へ

> **用語の予習・復習**: `docs/glossary.md` の Step 5 を参照。キーワードを見て自分で説明してみてから読むこと。

## このステップの到達目標

1. 分布仮説と共起行列から word2vec までの流れを説明できる
2. ハイパーパラメータ（window / dim / min_count / sg）の意味と影響を言える
3. 近傍語・類推・クラスタリングで word embedding を点検できる
4. **word embedding が不安定になる条件**を知り，結果を過信しない

## 導入：分布仮説

> "You shall know a word by the company it keeps." — J. R. Firth (1957)

語の意味は，その語が現れる文脈の分布で近似できる。これが分布仮説である。
word2vec はこの仮説を，**文脈語を予測する浅いニューラルネットの重み**として
実装したものにすぎない。学習が終わったあとに残る重み行列が word embedding である。

### 2つの学習方式

| | 入力 → 出力 | 特徴 |
|---|---|---|
| **CBOW** | 文脈語 → 中心語 | 速い。高頻度語に強い |
| **Skip-gram (sg=1)** | 中心語 → 文脈語 | 遅い。**低頻度語に強い** |

文学コーパスは総語数が数百万語と小さく，関心のある語（「恋」「自由」「機械」）も
必ずしも高頻度ではない。したがって **skip-gram を既定にする**。

## 参考
- Mikolov et al. (2013) Efficient estimation of word representations. *ICLR Workshop*.
- Levy & Goldberg (2014) Neural word embedding as implicit matrix factorization. *NIPS*.
- Antoniak & Mimno (2018) Evaluating the stability of embedding-based word similarities. *TACL* 6.


In [ ]:
# ---- 共通の準備（毎回このセルから実行する）----------------------------
import os, sys, csv, json, math, random, shutil, subprocess, warnings
import importlib.util
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.colors import LinearSegmentedColormap

warnings.filterwarnings('ignore', category=FutureWarning)

# リポジトリのルートを自動で探す（my_work/notebooks/ でも notebooks/ でも，上へたどる）
ROOT = Path.cwd()
while not (ROOT / 'config' / 'pipeline.yaml').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'scripts'))
print('ROOT =', ROOT)
if Path.cwd().resolve() == (ROOT / 'notebooks').resolve():
    print('[注意] 配布版（notebooks/）を直接開いている。実行すると次の git pull が止まる。\n'
          '       python scripts/copy_notebooks.py でコピーを作り，my_work/notebooks/ の方を開くこと。')

# 日本語フォント（□ にならないように）
for cand in ['Hiragino Sans', 'Yu Gothic', 'Meiryo',
             'Noto Sans CJK JP', 'IPAexGothic', 'MS Gothic']:
    if cand in {f.name for f in font_manager.fontManager.ttflist}:
        plt.rcParams['font.family'] = cand
        break
else:
    print('[!] 日本語フォントが見つかりません。docs/00_setup_students.md §1.7 を参照。')
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.dpi'] = 120

# ---- 図はすべて SVG（ベクタ）で保存する ---------------------------------
# 論文・スライドに載せる図は拡大しても劣化してはならない。PNG は解像度が
# 固定されるので，投影や印刷で文字が潰れる。SVG なら任意の倍率で鮮明で，
# Illustrator / Inkscape で軸ラベルだけを直すこともできる。
FIG_EXT   = 'svg'
RASTER_DPI = 200          # rasterized=True の要素にだけ効く
plt.rcParams['svg.fonttype']       = 'path'   # 文字をアウトライン化して環境非依存に
plt.rcParams['savefig.transparent'] = False
# 画面へのインライン表示は既定（PNG）のままにする。
# InlineBackend.figure_formats を 'svg' に変えると，JupyterLab や
# VS Code の版によっては図がまったく表示されなくなることがある。
# **保存されるファイルは SVG** なので，論文・スライドに使うほうは
# ベクタで手元に残る。画面で拡大して見たいときは save_fig が表示する
# パスの .svg をブラウザで開くこと。

def need(path, hint=''):
    """必要な入力があるか確かめる。無ければ**理由を表示して** False を返す。

    セルを `if p.exists():` で囲むと，入力が無いときに何も起きない。
    学生には「壊れている」と「まだ前の工程を走らせていない」の区別が
    つかず，図が出ないという相談の大半がこれである。必ず理由を出す。
    """
    p = Path(path)
    try:
        ok = p.is_file() or (p.is_dir() and any(p.iterdir()))
    except OSError:
        ok = False
    if not ok:
        print(f'[未実行] {p} がありません。')
        if hint:
            print(f'         {hint}')
        print('         この Step の前のセルを上から順に実行すること。'
              '\n         それでも出ない場合は，前の Step のノートブックが'
              '最後まで通っているか確認する。')
    return ok


# 旧名 → 新名。**中身は 0–1 の割合なので per cent は誤称**であった。
# 2026-09-24 に改名。古い出力を持っている人のために読み替えだけは残す。
LEGACY_COLS = {'df_all_pct': 'df_all_prop', 'df_in_pct': 'df_in_prop'}


def read_table(path, **kw):
    """CSV を読み，**古い列名があれば新しい名前に読み替える**。

    列名の約束：割合（0–1）は ``_prop`` / ``_ratio`` / ``_share``，
    百分率（0–100）だけを ``_pct`` と綴る。``df_all_prop`` が 0.1584 なら
    15.84 % の意である。読み替えたときは黙らずに知らせる — 黙って直すと，
    手元の CSV と教材の列名が食い違っていることに気づけないため。
    """
    d = pd.read_csv(path, **kw)
    old = {k: v for k, v in LEGACY_COLS.items()
           if k in d.columns and v not in d.columns}
    if old:
        d = d.rename(columns=old)
        print('[note] 古い列名を読み替えた: '
              + '，'.join(f'{k}→{v}' for k, v in old.items())
              + '\n       07_descriptive_stats.py を走らせ直すと'
                '新しい名前で書き出される。')
    return d


def load_meta(path=None, analysis_only=True):
    """メタデータを読む。既定では**分析に使う行だけ**を返す。

    落とすのは2種類。書誌としては残すが，集計に足してはいけない行である。
      superseded … v1 の合本。増補で分冊ごとに取り直したので，足すと
                   同じ作品を二重に数える
      merged     … 分冊。03b で canonical の巻に本文を統合したので，
                   この行はもう本文を持たない（『夜明け前』『家』）
      too_short  … 1チャンクにも満たず，チャンク単位の分析に乗らない

    生の表がほしいときは ``analysis_only=False``。
    """
    df = pd.read_csv(path or META)
    if analysis_only and 'completeness' in df.columns:
        drop = df['completeness'].isin(['superseded', 'merged', 'too_short'])
        if drop.any():
            names = '，'.join(df.loc[drop, 'title_aozora'].astype(str))
            print(f'[meta] 分析から除外 {int(drop.sum())} 行: {names}')
        df = df[~drop].reset_index(drop=True)
    return df


def w_ljust(text, width):
    """全角を2桁と数えて左詰めする。

    ``f'{s:<26}'`` は**文字数**で詰めるので，日本語の作品名を並べると
    桁が揃わない（全角は2桁ぶんの幅を占める）。表として読ませるなら
    表示幅で詰めること。

    **なお，一覧を出すなら ``show()`` で表にするほうがよい**（下記）。
    この関数は，表にしにくいもの（KWIC の前後文脈など）を print で
    並べるときに使う。
    """
    import unicodedata
    text = str(text)
    w = sum(2 if unicodedata.east_asian_width(c) in 'WF' else 1 for c in text)
    return text + ' ' * max(0, width - w)


# ---------------------------------------------------------------------------
# 分析結果の表示
# ---------------------------------------------------------------------------
# **一覧は print ではなく表で出す。**
#   * print は桁が揃わない（全角の幅）。数字の比較がしにくい
#   * 列に名前が付かないので，あとで見返したときに何の数字か分からない
#   * 並べ替えも絞り込みもできない
# 表にすると，列名がそのまま「何を測ったか」の記録になる。
# **ただし何でも表にするのではない。** 単発の数値・警告・KWIC の前後文脈は
# 文のほうが読みやすい。目安は「2列以上あるか」「行が並ぶか」。
TABLE_STYLES = [
    {'selector': 'caption',
     'props': [('caption-side', 'top'), ('text-align', 'left'),
               ('font-weight', '600'), ('padding', '0 0 .4em 0'),
               ('color', '#33322e'), ('font-size', '95%')]},
    {'selector': 'th',
     'props': [('background', '#f2f2ef'), ('text-align', 'left'),
               ('font-weight', '600'), ('padding', '.26em .7em'),
               ('border-bottom', '1px solid #c6c5bd'), ('white-space', 'nowrap')]},
    {'selector': 'td',
     'props': [('padding', '.22em .7em'), ('border-bottom', '1px solid #ecebe6')]},
    {'selector': 'tbody tr:hover td', 'props': [('background', '#f7f7f4')]},
]


def show(df, caption='', fmt=None, index=False, header=True, na='—', align=None):
    """DataFrame を表として表示する。Jupyter 以外でも落ちない。

    ``fmt`` は pandas の ``Styler.format`` に渡す辞書
    （例 ``{'一致率': '{:.1%}', 'G²': '{:.0f}'}``）。
    数値の列は自動で右寄せにする。``align`` で列ごとに寄せを指定できる。
    KWIC の左文脈を ``align={'左文脈': 'right'}`` にすると，
    **キーワードが縦に揃う**（等幅フォントに頼らずに揃う）。
    返り値は ``df`` なので ``t = show(df)`` として続けて使える。
    """
    if isinstance(df, pd.Series):
        df = df.to_frame()
    try:
        from IPython.display import display as _display
        st = df.style.format(fmt, na_rep=na) if fmt else df.style.format(na_rep=na)
        st = st.set_table_styles(TABLE_STYLES)
        num = list(df.select_dtypes('number').columns)
        if num:
            st = st.set_properties(subset=num, **{'text-align': 'right'})
        for col, side in (align or {}).items():
            if col in df.columns:
                st = st.set_properties(subset=[col],
                                       **{'text-align': side,
                                          'white-space': 'pre'})
        if caption:
            st = st.set_caption(caption)
        if not index:
            st = st.hide(axis='index')
        if not header:
            st = st.hide(axis='columns')
        _display(st)
    except Exception:                                   # noqa: BLE001
        # ノートブックの外（スクリプトから import したとき）でも読める形
        if caption:
            print(caption)
        print(df.to_string(index=index, header=header))
    return df


def grid(items, ncol=8, caption=''):
    """語の並びを ``ncol`` 列の表にして表示する。

    40 語を1行に流すと折り返しで読めない。列に切ると目で追える。
    順位が要るなら ``show()`` に順位列を付けた表を渡すこと。
    """
    items = [str(x) for x in items]
    rows = [items[i:i + ncol] for i in range(0, len(items), ncol)]
    rows = [r + [''] * (ncol - len(r)) for r in rows]
    t = pd.DataFrame(rows, columns=[f'_{i}' for i in range(ncol)])
    return show(t, caption=caption, header=False)


def work_rows(meta_df=None):
    """``work_stem`` からメタデータの行を引く辞書を作る。

    ``meta_df`` を省くと**分析対象外の行も含めた全件**から作る。
    表示用の名前は，分析から外した作品についても引けるほうがよい。

    **鍵の綴りに注意。** 青空文庫の作品 ID は索引では 0 埋めされていない
    （``1743``）が，本パイプラインのファイル名は6桁に 0 埋めしてある
    （``000119_001743``）。素朴に連結すると ``000119_1743`` となり，
    **1件も一致しない**。辞書は空振りしても例外を出さないので，
    誰の何だか分からないまま最後まで通ってしまう。両方の綴りを登録する。

    ``file_v1`` は増補 45 点では空である。``os.path.splitext(nan)`` は
    例外になるので，文字列であることを確かめてから使う。
    """
    if meta_df is None:
        meta_df = load_meta(analysis_only=False)
    d = {}
    for _, r in meta_df.iterrows():
        fv = r.get('file_v1')
        if isinstance(fv, str) and fv.strip():
            d[os.path.splitext(fv)[0]] = r
        pid = str(r.get('aozora_person_id') or '').strip()
        wid = str(r.get('aozora_work_id') or '').strip()
        if pid and wid and pid.lower() != 'nan' and wid.lower() != 'nan':
            for k in (f'{pid.zfill(6)}_{wid.zfill(6)}',
                      f'{pid}_{wid}', f'{pid.zfill(6)}_{wid}'):
                d[k] = r
    return d


def work_labels(meta_df=None, maxlen=12, with_year=False):
    """``work_stem`` → ``作者『作品』`` の対応表を返す。

    ``000119_001743`` と出されても誰の何だか分からない。距離の近い
    ペアを見るときに**どの作家のどの作品か**が分からなければ，
    「作家効果か時代効果か」という問いにそもそも答えられない。
    表示するときは必ずこれを通すこと。
    """
    out = {}
    for k, r in work_rows(meta_df).items():
        t = str(r.get('title_aozora') or '')
        lab = f"{r.get('author_ja', '?')}『{t[:maxlen]}』"
        if with_year and str(r.get('year_first') or '').strip():
            lab += f"({r['year_first']})"
        out[k] = lab
    return out


def attach_meta(df, cols, stem_col='work_stem', meta_df=None, quiet=False,
                fill_blank=True):
    """``df`` に足りないメタデータの列を，``work_stem`` から引いて補う。

    ``fill_blank=True``（既定）なら，**列はあるのに値が空**のセルも補う。
    列が無いより，列があって半分が空のほうが危ない。列が無ければ
    ``AttributeError`` で止まるが，値が空だと**図がそのまま描けてしまう**。
    2026-09-22 に 07 の突合が外れ，101 点のうち 62 点の ``year_first`` が
    空になった。図は描けたが，62 点が「初出年不明」の灰色で並んだ。
    値の空きも数えて報告し，ここで補えるものは補う。

    分析スクリプトの出力は，その分析に要る列しか書かない。
    ``09_doc2vec.py`` の ``work_vectors.csv`` に ``genre_main`` が無いのは
    その一例である。ノートブックで ``wv.genre_main`` と書けば
    ``AttributeError: 'DataFrame' object has no attribute 'genre_main'``
    になるが，**足りないのは列であって情報ではない**。
    メタデータ表には必ずあるのだから，ここで引いて補えばよい。

    出力 CSV の列構成に図の描画が依存するのは弱い。分析スクリプトを
    書き換えるたびに図が落ちる。図の側で「要る列を宣言して取りに行く」
    ほうが，どちらを先に走らせても通る。

    引けなかった列は空のまま残し ``[warn]`` を出す。図が落ちるより，
    「この軸は塗れなかった」と分かったうえで出るほうがよい。
    """
    df = df.copy()
    if stem_col not in df.columns:
        if not quiet:
            print(f'[warn] {stem_col} 列が無いので補完できない: {list(cols)}')
        for c in cols:
            if c not in df.columns:
                df[c] = ''
        return df

    rows = work_rows(meta_df)

    # **まず鍵が合っているかを見る。** 合っていなければ何も補えない。
    # 「1件も合わない」のはたいてい 0 埋めの綴り違いで，黙って通すと
    # 全部の軸が空のまま図になる。
    stems = df[stem_col].astype(str)
    found = stems.map(lambda s: s in rows)
    if not quiet and not found.all():
        n_miss = int((~found).sum())
        lv = 'FATAL' if found.sum() == 0 else 'warn '
        print(f'[{lv}] {stem_col} がメタデータと突合できない行が '
              f'{n_miss}/{len(df)} 件ある: '
              + '，'.join(stems[~found].head(4)))
        if found.sum() == 0:
            print('        **1件も合っていない。** 作品 ID の 0 埋めの'
                  '綴り違いを疑うこと（例 000119_1743 と 000119_001743）。')
            print('        この表を作ったスクリプトの鍵の作り方を直すこと。')

    def _blank(v):
        return v is None or str(v).strip().lower() in ('', 'nan', 'none')

    for c in cols:
        if c not in df.columns:
            vals = [(lambda r: '' if r is None or _blank(r.get(c))
                     else r.get(c))(rows.get(s)) for s in stems]
            df[c] = vals
            n = int(sum(1 for v in vals if str(v).strip()))
            if not quiet:
                mark = 'ok  ' if n == len(df) else 'warn'
                print(f'[{mark}] {c} をメタデータから補完: {n}/{len(df)} 件')
            continue

        if not fill_blank:
            continue
        # 列はある。空のセルだけを埋める。
        blank = df[c].map(_blank)
        if not blank.any():
            continue
        filled = 0
        vals = df[c].tolist()
        for i, (s, is_blank) in enumerate(zip(stems, blank)):
            if not is_blank:
                continue
            r = rows.get(s)
            if r is not None and not _blank(r.get(c)):
                vals[i] = r.get(c)
                filled += 1
        df[c] = vals
        if not quiet:
            left = int(sum(1 for v in vals if _blank(v)))
            mark = 'fix ' if left == 0 else 'warn'
            print(f'[{mark}] {c} は {int(blank.sum())}/{len(df)} 件が空だった'
                  f' → {filled} 件をメタデータから補完'
                  + ('' if left == 0 else f'（なお {left} 件が空）'))
            if left:
                print('        **その列で塗る図・集計は，この件数を'
                      '報告に書くこと。**')
    return df


def label_points(ax, xs, ys, texts, fontsize=8, color='#333333', pad=4,
                 leader='line', leader_min=13, crowd_r=26,
                 leader_color='#8a8a83'):
    """散布図の注記を，重ならない位置だけに置き，遠いものは引き出し線で結ぶ。

    素朴に ``ax.annotate(t, (x, y))`` と書くと，**注目すべき点ほど一箇所に
    固まる**ので注記が必ず重なって読めなくなる。文語標識の上位は
    どれも口語標識がほぼ 0 で，対数軸の右下隅に密集するのが典型である。

    そこで点の周囲を順に試し，他の注記とも他の点とも重ならず，かつ軸の
    内側に収まる位置があればそこに置く。どこにも置けない注記は**置かずに
    数だけ報告する**。読めない字を重ねるより，図の外（下の表）で番号から
    引くほうがよい。

    **離れた位置に置いた注記は，引き出し線で点と結ぶ。** 避けた結果として
    注記は点から離れるので，線が無いとどの点の名前なのか分からなくなる
    ——密集した領域では隣の点の名前だと読まれる。線があれば，遠くへ逃がす
    ことに副作用が無くなるので，**近くに空きが無い注記も置ける**ようになる
    （候補の輪を 24・30 ポイントまで広げてあるのはそのため）。

    ``leader``
        ``'line'``（既定）… 矢じりの無い細線で結ぶ。図版の慣例はこちら。
        6.5pt の文字に矢じりを付けると，印そのものを覆って点が読めなくなる
        ``'arrow'`` … 小さな矢じりを付ける
        ``'none'`` … 結ばない（従来どおり）
    ``leader_min``
        この距離（ポイント）より遠くに置いた注記を結ぶ。既定は 0，
        つまり**すべて結ぶ**。注記は必ず点から離れた位置に置かれるので，
        離れている以上「どの点の名前か」は線でしか確定しない。
        線を省くと，隣の点の名前だと読まれる余地が残る。
    ``crowd_r``
        注記の近くに**自分以外の点**がこの半径（ピクセル）内にあるかを
        見る。``leader_min`` を上げて線を減らしたときでも，
        紛れる相手が居る注記だけは必ず結ぶための保険である。

    表示座標で矩形の重なりを見るので，**軸の位置が確定してから**呼ぶ。
    ``fig.tight_layout()`` はこの関数より**前**に呼ぶこと（後で呼ぶと軸が
    動き，せっかく避けた位置がずれる）。戻り値は置けた注記の数。
    """
    from matplotlib.transforms import Bbox
    xs = np.asarray(xs, dtype=float)
    ys = np.asarray(ys, dtype=float)
    texts = list(texts)
    if len(texts) == 0:
        return 0
    # **長さが違えば黙って切り詰めずに止める。** zip は短いほうに合わせるので，
    # 座標だけを絞り込んで名前を絞り忘れると，先頭から順に**別の作品の名前**が
    # 貼られた図が，何の警告も出さずに出来上がる。これがいちばん重い事故である。
    if not (len(xs) == len(ys) == len(texts)):
        raise ValueError(
            f'label_points: 長さが違う（x={len(xs)}, y={len(ys)}, '
            f'名前={len(texts)}）。座標と名前を同じ添字で絞り込むこと。'
            'たとえば P[pick,0] と組むのは names[pick] であって names ではない。')
    fig = ax.figure
    fig.canvas.draw()
    ren = fig.canvas.get_renderer()
    axbb = ax.get_window_extent(renderer=ren)

    def pad_box(b, w=2.0, h=1.5):
        """注記の矩形に**絶対量の余白**を足す。

        倍率（``expanded(1.08, …)``）では足りない。1桁の数字は幅 8px ほど
        なので 8% は 0.6px にしかならず，隣り合う注記が触れるほど近くても
        「重なっていない」と判定される。**``21`` が「21」と読める**のは
        これが原因である。文字の大小によらず一定の余白を確保する。
        """
        return Bbox.from_extents(b.x0 - w, b.y0 - h, b.x1 + w, b.y1 + h)

    def bb_of(ann):
        # Annotation 自身の get_window_extent を使うこと。
        # Text.get_window_extent(ann, ...) を呼ぶと xy の位置が無視され，
        # xytext のオフセットを絶対座標と見た矩形が返って判定が壊れる。
        return ann.get_window_extent(renderer=ren)

    blocked = []
    for coll in ax.collections:
        try:
            for p in coll.get_offsets():
                px, py = ax.transData.transform(p)
                blocked.append(Bbox.from_bounds(px - pad, py - pad,
                                                2 * pad, 2 * pad))
        except Exception:                                    # noqa: BLE001
            pass

    # **まっすぐ真上・真下を先に試す。** 点の直上に中央揃えで置ければ，
    # それがいちばん素直で，引き出し線も要らない。横へずらすのは，
    # 直上が塞がっていたときの次善である。
    #
    # 横へずらす輪は 12 ポイントから始める。**線が線として見える長さを
    # 確保する**ため。8 ポイントに置くと引き出し線が3ピクセルの点にしか
    # ならず，汚れと区別がつかない。
    # **真上に置けなければ，まず真上へ逃がす。** 横へ逃がすと注記の左右の
    # 順序が点の順序と入れ替わり，引き出し線も交差する。真上に段を重ねる
    # 限り，x は動かないので順序は必ず保たれる。横へずらすのは最後。
    # **横のずらし幅は小さく取る。** 横へ 30 ポイントも動かすと，注記が
    # 隣の点の真上に乗り，引き出し線で結んでも読みにくい。真上に段を
    # 重ねるほうが先で（x が動かないので順序が保たれる），横は 8→18
    # ポイントの範囲に収める。
    CAND = [(0, 9), (0, -11), (0, 20), (0, -22), (0, 31), (0, -33),
            (8, 5), (-8, 5), (8, -12), (-8, -12),
            (11, 0), (-11, 0),
            (13, 9), (-13, 9), (13, -16), (-13, -16),
            (18, 0), (-18, 0), (18, 14), (-18, 14),
            (0, 42), (0, -44)]

    def _ha(dx):
        # dx が 0 なら**中央揃え**。ここを 'left' にすると，真上に置いた
        # つもりの注記が文字幅の半分だけ右にずれ，隣の点の上に乗る。
        return 'center' if dx == 0 else ('left' if dx > 0 else 'right')
    placed, chosen, skipped = [], [], 0
    for x, y, t in zip(xs, ys, texts):
        # **自分が指している点は避けない。** 除かないと，注記は必ず
        # 自分の点の隣に来るので全部「重なる」と判定され，1つも置けない。
        ox, oy = ax.transData.transform((x, y))
        near = [b for b in blocked
                if not (abs((b.x0 + b.x1) / 2 - ox) < 1
                        and abs((b.y0 + b.y1) / 2 - oy) < 1)]
        for dx, dy in CAND:
            ann = ax.annotate(str(t), (x, y), textcoords='offset points',
                              xytext=(dx, dy), fontsize=fontsize, color=color,
                              ha=_ha(dx),
                              va='bottom' if dy >= 0 else 'top', zorder=6)
            bb = pad_box(bb_of(ann))
            inside = (bb.x0 >= axbb.x0 and bb.x1 <= axbb.x1
                      and bb.y0 >= axbb.y0 and bb.y1 <= axbb.y1)
            if inside and not any(bb.overlaps(b) for b in placed + near):
                placed.append(bb)
                chosen.append((ann, x, y, str(t), dx, dy))
                break
            ann.remove()
        else:
            skipped += 1

    # ---- 交差をほどく ----------------------------------------------------
    # **引き出し線が交差すると，注記の左右の順序が点の順序と入れ替わる。**
    # 文語標識の上位のように順位そのものが意味を持つ図では，2 と 3 が
    # 入れ替わって並ぶだけで読み違えられる。交差している2件を見つけ，
    # **位置を入れ替えて交差が解ければ入れ替える**（2-opt）。
    def _cross(p, q, r, s):
        def o(a, b, c):
            return ((b[0] - a[0]) * (c[1] - a[1])
                    - (b[1] - a[1]) * (c[0] - a[0]))
        return (((o(r, s, p) > 0) != (o(r, s, q) > 0))
                and ((o(p, q, r) > 0) != (o(p, q, s) > 0)))

    kpt = fig.dpi / 72.0

    def _seg(i):
        ann, x, y, t, dx, dy = chosen[i]
        ox, oy = ax.transData.transform((x, y))
        return (ox, oy), (ox + dx * kpt, oy + dy * kpt)

    def _set_off(i, dx, dy):
        ann, x, y, t, _, _ = chosen[i]
        ann.set_position((dx, dy))
        ann.set_ha(_ha(dx))
        ann.set_va('bottom' if dy >= 0 else 'top')
        chosen[i] = (ann, x, y, t, dx, dy)

    def _fits(i, bb):
        ox, oy = ax.transData.transform((chosen[i][1], chosen[i][2]))
        if not (bb.x0 >= axbb.x0 and bb.x1 <= axbb.x1
                and bb.y0 >= axbb.y0 and bb.y1 <= axbb.y1):
            return False
        return not any(bb.overlaps(b) for b in blocked
                       if abs((b.x0 + b.x1) / 2 - ox) > 1
                       or abs((b.y0 + b.y1) / 2 - oy) > 1)

    swaps = 0
    for _ in range(3):
        improved = False
        for i in range(len(chosen)):
            for j in range(i + 1, len(chosen)):
                if not _cross(*_seg(i), *_seg(j)):
                    continue
                di, dj = chosen[i][4:6], chosen[j][4:6]
                _set_off(i, *dj)
                _set_off(j, *di)
                bi = pad_box(bb_of(chosen[i][0]))
                bj = pad_box(bb_of(chosen[j][0]))
                others = [b for k2, b in enumerate(placed) if k2 not in (i, j)]
                good = (not bi.overlaps(bj)
                        and not any(bi.overlaps(b) or bj.overlaps(b)
                                    for b in others)
                        and _fits(i, bi) and _fits(j, bj)
                        and not _cross(*_seg(i), *_seg(j)))
                if good:
                    placed[i], placed[j] = bi, bj
                    swaps += 1
                    improved = True
                    continue
                _set_off(i, *di)
                _set_off(j, *dj)

                # 入れ替えが収まらないときは，**片方を別の候補位置へ動かす**。
                # 入れ替えは2つの箱の大きさが違うと失敗しやすい（数字1桁と
                # 作者名では幅が違う）。動かすほうは箱の大きさが変わらない。
                moved = False
                for who, other in ((i, j), (j, i)):
                    d0 = chosen[who][4:6]
                    for cx, cy in CAND:
                        if (cx, cy) == tuple(d0):
                            continue
                        _set_off(who, cx, cy)
                        bw = pad_box(bb_of(chosen[who][0]))
                        rest = [b for k2, b in enumerate(placed) if k2 != who]
                        if (_fits(who, bw)
                                and not any(bw.overlaps(b) for b in rest)
                                and not _cross(*_seg(who), *_seg(other))
                                and not any(_cross(*_seg(who), *_seg(k2))
                                            for k2 in range(len(chosen))
                                            if k2 != who)):
                            placed[who] = bw
                            swaps += 1
                            moved = improved = True
                            break
                        _set_off(who, *d0)
                    if moved:
                        break
        if not improved:
            break

    # ---- 引き出し線 ------------------------------------------------------
    # **線は配置が全部決まってから付ける。** arrowprops を付けた
    # Annotation の get_window_extent は「文字＋線」の外接矩形を返すので，
    # 配置の判定に使うと自分の点と必ず重なり，1件も置けなくなる。
    n_leader = 0
    if leader in ('line', 'arrow'):
        style = '-' if leader == 'line' else '-|>'
        for (ann, x, y, t, dx, dy), bb in zip(chosen, placed):
            ox, oy = ax.transData.transform((x, y))

            def dist_to_box(px, py, b=bb):
                # 文字の矩形から点までの距離。矩形の中なら 0。
                ddx = max(b.x0 - px, 0, px - b.x1)
                ddy = max(b.y0 - py, 0, py - b.y1)
                return (ddx * ddx + ddy * ddy) ** .5

            # **素直に置けたものには線を引かない。** 点の直上（または直下）に
            # 中央揃えで載っていて，しかもその注記にいちばん近い点が自分の
            # 点であれば，どの点の名前かは見れば分かる。線はかえって邪魔
            # である。横へ逃がしたものだけを結ぶ。
            if dx == 0 and abs(dy) <= 12:
                continue            # 点の真上・真下の一段目 → 線は要らない
            # それ以外は結ぶ。**段を上げたものも結ぶ。** 一段上げた注記の
            # 真下には別の点の注記が入るので，どちらの点のものか分からなく
            # なる。横へずらしたものは言うまでもない。
            d_other = min(
                (dist_to_box((b.x0 + b.x1) / 2, (b.y0 + b.y1) / 2)
                 for b in blocked
                 if abs((b.x0 + b.x1) / 2 - ox) > 1
                 or abs((b.y0 + b.y1) / 2 - oy) > 1),
                default=float('inf'))
            if (dx * dx + dy * dy) ** .5 < leader_min and d_other >= crowd_r:
                continue
            ann.remove()
            ax.annotate(t, (x, y), textcoords='offset points',
                        xytext=(dx, dy), fontsize=fontsize, color=color,
                        ha=_ha(dx),
                        va='bottom' if dy >= 0 else 'top', zorder=6,
                        arrowprops=dict(arrowstyle=style, linewidth=.55,
                                        color=leader_color, alpha=.9,
                                        shrinkA=1.5, shrinkB=2.5,
                                        mutation_scale=7))
            n_leader += 1

    # ---- 誤読の自己点検 --------------------------------------------------
    # **注記の最寄りの点が自分の点でないものを数える。** これが
    # 「ラベルとデータ点がずれて見える」の正体である。引き出し線を
    # 引いてあれば誤読にはならないが，線を切った設定では危険なので，
    # そのときだけ警告を出す。
    risky = []
    for (ann, x, y, t, dx, dy), bb in zip(chosen, placed):
        ox, oy = ax.transData.transform((x, y))
        cx, cy = (bb.x0 + bb.x1) / 2, (bb.y0 + bb.y1) / 2
        d_own = ((cx - ox) ** 2 + (cy - oy) ** 2) ** .5
        d_other = min(
            (((b.x0 + b.x1) / 2 - cx) ** 2 + ((b.y0 + b.y1) / 2 - cy) ** 2) ** .5
            for b in blocked
            if abs((b.x0 + b.x1) / 2 - ox) > 1 or abs((b.y0 + b.y1) / 2 - oy) > 1
        ) if len(blocked) > 1 else float('inf')
        if d_other < d_own:
            risky.append(t)
    left = sum(1 for i in range(len(chosen)) for j in range(i + 1, len(chosen))
               if _cross(*_seg(i), *_seg(j)))
    if skipped:
        print(f'[fig] 重なるため {skipped} 件の注記を省いた（表で引くこと）')
    if left:
        print(f'[warn] 引き出し線の交差が {left} 件ほどけなかった。'
              '注記の左右の順序が点の順序と食い違う。'
              '注記を短くするか，件数を減らすこと。')
    if risky:
        head = '，'.join(str(r) for r in risky[:6])
        more = f' ほか{len(risky) - 6}件' if len(risky) > 6 else ''
        if leader in ('line', 'arrow'):
            print(f'[fig] {len(risky)} 件の注記は別の点のほうが近い'
                  f'（{head}{more}）。引き出し線で結んであるので読み違えない。')
        else:
            print(f'[warn] {len(risky)} 件の注記は**別の点のほうが近い**'
                  f'（{head}{more}）。leader="none" では読み違えが起きる。')
    return len(placed)
def _proj_versions():
    """射影に関わる版を並べる（うまくいかないときの手がかり）。"""
    import importlib
    out = []
    for nm in ['numpy', 'numba', 'llvmlite', 'pynndescent', 'sklearn']:
        try:
            out.append(f'{nm} ' + str(getattr(importlib.import_module(nm),
                                              '__version__', '?')))
        except Exception:                                    # noqa: BLE001
            out.append(f'{nm} ×')
    return '／'.join(out)

def umap_diagnosis(e):
    """UMAP が使えないときに，**何をすればよいか**を出す。"""
    import sys
    print(f'[NG  ] UMAP が使えない: {type(e).__name__}: {e}')
    print(f'       このカーネルの Python = {sys.executable}')
    print(f'       {_proj_versions()}')
    if isinstance(e, ModuleNotFoundError):
        # **入れた先とカーネルの環境が違う**のが圧倒的に多い。
        # uv add は「プロジェクト（pyproject.toml のある場所）」単位なので，
        # dh_project/pyproject.toml が無い，または dh_project の外に clone
        # した場合は，uv は別のプロジェクトに入れる。カーネルの .venv には入らない。
        print('       **この環境には入っていない。** 入れた先が違う可能性が高い')
        print('       （uv add はプロジェクト単位。~/Documents/dh_project に')
        print('        pyproject.toml が無いと，別のプロジェクトに入る）。')
        print('       この環境を名指しして入れるのが確実:')
        import platform as _pf
        if sys.platform == 'darwin' and _pf.machine() == 'x86_64':
            # **Intel Mac は版を固定する。** llvmlite の x86_64 wheel は
            # 0.45.1 が最後で，0.46 以降は arm64 のみ。固定しないと
            # ソースからのビルドに落ち，Homebrew の LLVM と版が合わずに
            # 失敗する（llvmlite 0.49 は LLVM 22 を要求）。
            print('       （Intel Mac なので**版を固定する**。'
                  'llvmlite の x86_64 wheel は 0.45.1 が最後）')
            print(f'         uv pip install --python "{sys.executable}" \\')
            print('             --only-binary :all: \\')
            print('             "numba==0.62.1" "llvmlite==0.45.1" '
                  '"numpy<2.4" umap-learn')
        else:
            print(f'         uv pip install --python "{sys.executable}" umap-learn')
        print('       入れたら**カーネルを再起動**して，このセルから実行し直す。')
    else:
        print('       import は通るが使えない型の失敗である'
              '（別パッケージの umap／numba と numpy の版違い／'
              'numba のキャッシュ）。')
    print('       切り分けの全項目:')
    print('         import sys, subprocess; print(subprocess.run('
          '[sys.executable,')
    print("             str(ROOT/'scripts'/'check_umap.py')], "
          'capture_output=True,')
    print('             text=True).stdout)')

def project(Xn, how='umap', seed=20260920, n_neighbors=15, min_dist=0.12,
            perplexity=30):
    """高次元の行列を2次元に落とす。**どの方法で落としたかを必ず返す。**

    ``how`` は ``'umap'``／``'tsne'``／``'auto'``。既定の ``'umap'`` は，
    使えなければ**止まって理由を出す**。``'auto'`` のときだけ t-SNE に落ちる。
    **黙って別の方法に替えないのが肝心である**（図は出るが塊の見え方は
    変わるので，環境の問題を分析結果と読み違える）。

    Step 5 の §3（主成分分析との比較）と §4（ギャラクシー）が共有する。
    """
    import importlib
    Xn = np.asarray(Xn, dtype=np.float32)
    if how in ('auto', 'umap'):
        try:
            m = importlib.import_module('umap')
            if not hasattr(m, 'UMAP'):
                # PyPI には umap（別物）と umap-learn（本物）がある。
                # pip install umap をしていると import umap はそちらを拾う。
                raise ImportError(
                    f'umap に UMAP クラスが無い（{getattr(m, "__file__", "?")}）。'
                    '別パッケージの umap が入っている。'
                    'umap を外して umap-learn を入れること')
            P = m.UMAP(n_neighbors=n_neighbors, min_dist=min_dist,
                       metric='cosine', random_state=seed).fit_transform(Xn)
            return (np.asarray(P, dtype=np.float32),
                    f'UMAP {getattr(m, "__version__", "")}'
                    f' (n_neighbors={n_neighbors}, min_dist={min_dist}, cosine)')
        except Exception as e:                               # noqa: BLE001
            umap_diagnosis(e)
            if how == 'umap':
                # **黙って別の方法に替えない。** どうしても t-SNE で
                # 進めたいときは 'tsne' と明示し，報告にもそう書くこと。
                raise
            print('[warn] how="auto" なので t-SNE に切り替える。'
                  '**図と報告に t-SNE と書くこと。**')
    from sklearn.manifold import TSNE
    P = TSNE(n_components=2, perplexity=perplexity, metric='cosine',
             init='pca', random_state=seed).fit_transform(Xn)
    return (np.asarray(P, dtype=np.float32),
            f't-SNE (perplexity={perplexity}, cosine)')


def proj_quality(Xn, P, k=10):
    """射影がどれだけ嘘をついているかを3つの数で返す。

    ``trust``  … 2次元で近く見える点が原空間でも近いか（局所・1が最良）
    ``keep``   … 原空間の上位 k 近傍のうち画面でも上位 k に入る語数
    ``rho``    … 原空間の距離と画面の距離の順位相関（**大域**の保存）

    局所（trust・keep）と大域（rho）は別物である。**UMAP は局所に強く，
    主成分分析は大域に強い**——これを目で見ずに数で確かめるための関数。
    """
    from scipy.spatial.distance import pdist
    from scipy.stats import spearmanr
    from sklearn.manifold import trustworthiness
    Xn, P = np.asarray(Xn, np.float32), np.asarray(P, np.float32)
    S = Xn @ Xn.T
    np.fill_diagonal(S, -np.inf)
    nn_t = np.argsort(-S, axis=1)[:, :k]
    d2 = ((P[:, None, :] - P[None, :, :]) ** 2).sum(-1)
    np.fill_diagonal(d2, np.inf)
    nn_p = np.argsort(d2, axis=1)[:, :k]
    keep = np.array([len(set(a) & set(b)) for a, b in zip(nn_t, nn_p)])
    trust = float(trustworthiness(Xn, P, n_neighbors=k, metric='cosine'))
    rho = float(spearmanr(pdist(Xn, 'cosine'), pdist(P))[0])
    return {'trust': trust, 'keep': keep, 'rho': rho}


def reserve_right(fig, frac=0.80):
    """面の外に凡例を置いた図で，**右に余白を確保する**。

    ``tight_layout()`` は面の外に置いた凡例を数えないので，そのままだと
    凡例が図の枠から出る。静止版は ``bbox_inches='tight'`` で救われるが，
    **HTML に埋め込む版は切り取らない**（切り取ると点の位置の割合が
    ずれる）ので，凡例が切れて読めなくなる。実際に切れた。

    ``tight_layout()`` の**後**，注記（``label_points``）の**前**に呼ぶ。
    """
    fig.subplots_adjust(right=frac)


def save_fig(fig, stem, out=None):
    """図を SVG で保存してパスを表示する。

    stem は拡張子なしの名前（例 'Step1_period_balance'）。
    点が数千個ある散布図は，散布図だけ rasterized=True にしておくと
    軸と文字はベクタのままファイルが軽くなる。
    """
    d = Path(out) if out else OUT
    d.mkdir(parents=True, exist_ok=True)
    path = d / f'{stem}.{FIG_EXT}'
    fig.savefig(path, format=FIG_EXT, dpi=RASTER_DPI, bbox_inches='tight')
    print(f'[fig] {path}  ({path.stat().st_size/1024:,.0f} KB)')
    return path


# ---------------------------------------------------------------------------
# 対話的な図（SVG はそのまま残す）
# ---------------------------------------------------------------------------
# 散布図の点が何百個あると，注記を付けられるのはごく一部である。残りの点は
# 「どの語か」が分からないまま眺めることになる。かといって全点に名前を
# 付ければ図は読めない。
#
# そこで**同じ図から2つ出す**。
#   * ``<stem>.svg``  … 論文・配布用。これまでどおり。加筆も拡大も自由
#   * ``<stem>.html`` … 授業・探索用。SVG をそのまま埋め込み，
#                       その上に当たり判定を重ねて，指した点の語を出す
#
# **HTML は SVG を作り直さない。同じ SVG を中に入れる。** 別に描き直すと
# 図が2種類できて，どちらが正かが分からなくなる。注記（bursty な語の
# ラベル）も SVG の中にあるのでそのまま残る。
#
# 外部の JS ライブラリは使わない。CDN が塞がれた機体でも開けるようにする。
INTERACTIVE_CSS = """
:root { --ink:#1f1e1b; --ink2:#5a5a55; --line:#d8d7d0; --surface:#ffffff;
        --wash:#f7f7f4; --accent:#184f95; }
* { box-sizing:border-box; }
body { margin:0; padding:24px 16px 48px; background:var(--wash);
       color:var(--ink); font-family:"Hiragino Sans","Noto Sans JP",
       "Yu Gothic",system-ui,sans-serif; line-height:1.6; }
.wrap { max-width:1100px; margin:0 auto; }
h1 { font-size:1.15rem; margin:0 0 .2em; font-weight:650; }
.sub { color:var(--ink2); font-size:.86rem; margin:0 0 1.1em; }
.card { background:var(--surface); border:1px solid var(--line);
        border-radius:10px; padding:14px; }
.figbox { position:relative; }
.figbox svg { width:100%; height:auto; display:block; }
#hit { position:absolute; inset:0; cursor:crosshair; }
#ring { position:absolute; width:22px; height:22px; margin:-11px 0 0 -11px;
        border:2px solid var(--accent); border-radius:50%;
        pointer-events:none; opacity:0; transition:opacity .08s; }
#tip { position:absolute; z-index:5; min-width:190px; max-width:290px;
       background:var(--surface); border:1px solid var(--line);
       border-radius:8px; box-shadow:0 6px 20px rgba(0,0,0,.13);
       padding:9px 11px; font-size:.8rem; pointer-events:none; opacity:0;
       transition:opacity .08s; }
#tip .term { font-size:1.05rem; font-weight:650; letter-spacing:.02em;
             margin-bottom:.35em; word-break:break-all; }
#tip dl { display:grid; grid-template-columns:auto 1fr; gap:1px 10px;
          margin:0; }
#tip dt { color:var(--ink2); font-size:.74rem; white-space:nowrap; }
#tip dd { margin:0; text-align:right; font-variant-numeric:tabular-nums;
          font-weight:600; }
.bar { display:flex; gap:10px; align-items:center; flex-wrap:wrap;
       margin:14px 0 0; font-size:.82rem; color:var(--ink2); }
.bar input { font:inherit; padding:5px 9px; border:1px solid var(--line);
             border-radius:6px; min-width:190px; background:var(--surface); }
.bar a { color:var(--accent); }
table { border-collapse:collapse; width:100%; font-size:.78rem;
        margin-top:10px; }
th,td { padding:4px 8px; border-bottom:1px solid #ecebe6; text-align:left;
        white-space:nowrap; }
th { background:var(--wash); position:sticky; top:0; font-weight:650; }
td.num { text-align:right; font-variant-numeric:tabular-nums; }
tbody tr:hover td { background:var(--wash); }
tbody tr.on td { background:#eaf1fb; }
.scroll { max-height:340px; overflow:auto; border:1px solid var(--line);
          border-radius:8px; margin-top:10px; }
.hint { font-size:.78rem; color:var(--ink2); margin:.6em 0 0; }
#links { position:absolute; inset:0; pointer-events:none; overflow:visible; }
#links line { stroke:var(--accent); stroke-width:1.1; opacity:.55; }
#links circle { fill:none; stroke:var(--accent); stroke-width:1.4; opacity:.8; }
#marks { position:absolute; inset:0; pointer-events:none; overflow:visible; }
#marks circle { fill:none; stroke:#d55e00; stroke-width:1.6; opacity:.9; }
#tip .notes { margin:.45em 0 0; font-size:.76rem; color:var(--ink);
              border-top:1px solid var(--line); padding-top:.4em;
              line-height:1.5; word-break:break-all; }
#tip .notes b { color:var(--ink2); font-weight:600; }
.prov { font-size:.72rem; color:var(--ink2); margin:.9em 0 0;
        border-top:1px solid var(--line); padding-top:.6em;
        font-variant-numeric:tabular-nums; }
.danger { background:#fdf0ea; border:1px solid #e8a37c; border-radius:8px;
          padding:9px 12px; font-size:.85rem; color:#8a3b10;
          margin:0 0 12px; }
"""

INTERACTIVE_JS = r"""
// 点は data-* ではなく JSON で渡す。語はコーパス由来の任意の文字列なので，
// **HTML に文字列連結で差し込まない**（textContent で入れる）。
// 見出し（keys・nhead）は全点で同じなら1回だけ入っている。点が1万個ある
// 図では，これで HTML が 1 MB 以上軽くなる。古い形（配列だけ）も読む。
const RAW = JSON.parse(document.getElementById('pts-data').textContent);
const PTS = Array.isArray(RAW) ? RAW : RAW.pts;
const KEYS = (RAW && RAW.keys) || [];
const NHEAD = (RAW && RAW.nhead) || '';
const LINKNOTES = !!(RAW && RAW.linknotes);
function pairsOf(p) {
  if (p.fields) return p.fields;
  if (p.v) return p.v.map((x, i) => [KEYS[i] || '', x]);
  return [];
}
function notesOf(p) {
  if (p.notes) return p.notes;
  if (p.n) return [NHEAD, p.n];
  // 本文が無く linknotes が立っているときは，線で結ぶ先の語を並べる
  if (LINKNOTES && p.links && p.links.length) {
    return [NHEAD, p.links.map(j => (PTS[j] || {}).term || '').join(' ')];
  }
  return null;
}
const box = document.getElementById('hit');
const tip = document.getElementById('tip');
const ring = document.getElementById('ring');
const rows = Array.from(document.querySelectorAll('tbody tr'));
const links = document.getElementById('links');
const marks = document.getElementById('marks');

// **最も近い点を拾う。** 点の直径は数ピクセルしかないので，
// 「真上に置く」ことを要求すると誰も当てられない（dataviz の規則）。
// カーソルに最も近い点を選び，遠すぎるときだけ何も出さない。
function nearest(px, py, w, h) {
  let best = null, bd = 1e9;
  for (const p of PTS) {
    const dx = p.x * w - px, dy = p.y * h - py;
    const d = dx * dx + dy * dy;
    if (d < bd) { bd = d; best = p; }
  }
  return Math.sqrt(bd) <= 34 ? best : null;   // 34px より遠ければ出さない
}

function fill(p) {
  tip.textContent = '';
  const h = document.createElement('div');
  h.className = 'term';
  h.textContent = p.term;                     // ← 連結しない
  tip.appendChild(h);
  const dl = document.createElement('dl');
  for (const [k, v] of pairsOf(p)) {
    const dt = document.createElement('dt'); dt.textContent = k;
    const dd = document.createElement('dd'); dd.textContent = v;
    dl.appendChild(dt); dl.appendChild(dd);
  }
  tip.appendChild(dl);
  const nt = notesOf(p);
  if (nt) {                                   // 近傍語など，横に長い情報
    const n = document.createElement('p');
    n.className = 'notes';
    const b = document.createElement('b');
    b.textContent = nt[0] + ' ';
    n.appendChild(b);
    n.appendChild(document.createTextNode(nt[1]));
    tip.appendChild(n);
  }
}

// **原空間での近傍を線で結ぶ。** 画面の近さは射影の結果にすぎない。
// 線が遠くへ伸びるなら，その点の近傍関係は2次元に収まっていない。
// これを見せるのが，この図でいちばん大事なところである。
function drawLinks(p, w, h) {
  if (!links) return;
  while (links.firstChild) links.removeChild(links.firstChild);
  if (!p.links || !p.links.length) return;
  const NS = 'http://www.w3.org/2000/svg';
  for (const j of p.links) {
    const q = PTS[j];
    if (!q) continue;
    const ln = document.createElementNS(NS, 'line');
    ln.setAttribute('x1', p.x * w); ln.setAttribute('y1', p.y * h);
    ln.setAttribute('x2', q.x * w); ln.setAttribute('y2', q.y * h);
    links.appendChild(ln);
    const c = document.createElementNS(NS, 'circle');
    c.setAttribute('cx', q.x * w); c.setAttribute('cy', q.y * h);
    c.setAttribute('r', 5);
    links.appendChild(c);
  }
}

let cur = null;
function show(p, px, py) {
  const w = box.clientWidth, h = box.clientHeight;
  if (p !== cur) { fill(p); drawLinks(p, w, h); cur = p; }
  ring.style.left = (p.x * w) + 'px';
  ring.style.top = (p.y * h) + 'px';
  ring.style.opacity = 1;
  tip.style.opacity = 1;
  // はみ出さないように寄せる
  const tw = tip.offsetWidth, th = tip.offsetHeight;
  let lx = px + 16, ly = py + 14;
  if (lx + tw > w) lx = px - tw - 16;
  if (ly + th > h) ly = py - th - 14;
  tip.style.left = Math.max(0, lx) + 'px';
  tip.style.top = Math.max(0, ly) + 'px';
  rows.forEach(r => r.classList.toggle('on', r.dataset.i === String(p.r)));
}
function hide() {
  tip.style.opacity = 0; ring.style.opacity = 0; cur = null;
  if (links) while (links.firstChild) links.removeChild(links.firstChild);
  rows.forEach(r => r.classList.remove('on'));
}

box.addEventListener('pointermove', e => {
  const r = box.getBoundingClientRect();
  const p = nearest(e.clientX - r.left, e.clientY - r.top, r.width, r.height);
  if (p) show(p, e.clientX - r.left, e.clientY - r.top); else hide();
});
box.addEventListener('pointerleave', hide);

// 表の行にカーソルを乗せても，図の上の点が光る（逆引き）。
// **カーソルが使えない人にも同じ情報が届くように**，表を必ず添える
// （点が数千を超える図だけは表を絞る。絞ったことは図の下に明記する）。
rows.forEach(r => {
  r.addEventListener('mouseenter', () => {
    // 表の行は論理点。図の上では**先頭の面**の点を光らせる
    const p = PTS[Number(r.dataset.i)];
    if (!p) return;
    const w = box.clientWidth, h = box.clientHeight;
    show(p, p.x * w, p.y * h);
  });
  r.addEventListener('mouseleave', hide);
});

// 絞り込み。語・作品・時代のどれでも当たる
const q = document.getElementById('q');
if (q) q.addEventListener('input', () => {
  const s = q.value.trim();
  let n = 0;
  rows.forEach(r => {
    const hit = !s || r.textContent.includes(s);
    r.style.display = hit ? '' : 'none';
    if (hit) n++;
  });
  // 表を絞った図では，**表に無い語も図の上では当たる**。
  // 表の件数だけを出すと「無い」と誤解されるので両方を出す。
  const nlog = Number(document.body.dataset.nlog || rows.length);
  let extra = '';
  if (s && rows.length < nlog) {
    const seen = new Set();
    for (const p of PTS) if (p.term.includes(s)) seen.add(p.r);
    extra = '（図の上 ' + seen.size + ' 件）';
  }
  document.getElementById('count').textContent = n + ' 件' + extra;
  // **図の上にも印を付ける。** 表だけ絞っても「どこにあるか」は分からない。
  if (!marks) return;
  while (marks.firstChild) marks.removeChild(marks.firstChild);
  if (!s) return;
  const NS = 'http://www.w3.org/2000/svg';
  const w = box.clientWidth, h = box.clientHeight;
  let drawn = 0;
  for (const p of PTS) {
    if (!p.term.includes(s)) continue;
    const c = document.createElementNS(NS, 'circle');
    c.setAttribute('cx', p.x * w); c.setAttribute('cy', p.y * h);
    c.setAttribute('r', 7);
    marks.appendChild(c);
    if (++drawn > 400) break;          // 印が多すぎると図が読めない
  }
});
"""


def save_interactive(fig, ax, stem, xs, ys, tips, out=None, title='',
                     note='', table_cols=None, source=None, id_col='語',
                     hint='', table_idx=None, coords=None):
    """SVG を保存し，**同じ SVG を埋め込んだ対話的な HTML** も書く。

    ``xs`` ``ys`` はデータ座標，``tips`` は点ごとの情報
    （``{'term': 語, 'fields': [(見出し, 値), …]}`` の並び）。
    3つの長さは一致していなければならない。ずれたまま描くと，
    **指した点と出る語が食い違う**（注記の添字ずれと同じ事故）。

    位置は「図全体に対する割合」で書き出す。SVG を ``width:100%`` で
    伸縮させても割合は変わらないので，どんな幅でも点と当たり判定が
    合う。座標は matplotlib の変換を通して得るので，**図と HTML で
    座標の計算が二重にならない**。

    ``source`` に入力ファイルのパスを渡すこと。**どの表から描いた図かを
    HTML の末尾に刻む。** これが無いと，試験用の作りかけのデータから
    描いた図と，本番のデータから描いた図が見分けられない。
    入力がプロジェクトの外（``/tmp`` など）にあるときは
    「試験用」と赤字で出し，配布してはいけないことを図自身に言わせる。

    ``id_col`` は表の第1列の見出し（既定「語」。作品を点にする図では
    「作品」などに変える）。

    ``ax`` には**面の並び**も渡せる（``[axes[0], axes[1]]``）。同じ点を
    別の塗り分けで2面に描いた図では，どちらの面を指しても同じ情報が出る。
    表の行は点ごとに1行だけ作る（面の数だけ重複させない）。

    ``table_idx`` は**表に載せる点の添字**（既定は全点）。点が数千を超える
    図では表を全件出すと HTML が数 MB になり，読む側にも役に立たない。
    そのときは載せる点を選ぶ。**ただし図の当たり判定と検索は全点に効く**
    ので，表に無い語も指せるし検索で図に印が付く。表を絞ったときは，
    何件のうち何件を載せたかを HTML に明記する（黙って捨てないこと）。

    ``coords`` は**面ごとの座標**（``[(x1, y1), (x2, y2)]``）。同じ点を
    **違う座標系**で2面に描いた図（主成分分析と UMAP の比較など）で使う。
    渡さなければ全部の面で ``xs`` ``ys`` を使う。
    ⚠ 面ごとに座標が違うのに ``coords`` を渡さないと，2面めの当たり判定が
    1面めの座標で置かれる。**図は出るが，指した点と出る語が食い違う。**
    """
    import json as _json
    if not (len(xs) == len(ys) == len(tips)):
        raise ValueError(
            f'save_interactive: 長さが違う（x={len(xs)}, y={len(ys)}, '
            f'情報={len(tips)}）。座標と情報を同じ添字で絞り込むこと。')

    svg_path = save_fig(fig, stem, out=out)
    outdir = svg_path.parent

    # ---- 埋め込む SVG は**切り取らずに**保存する ------------------------
    # save_fig は bbox_inches='tight' で余白を詰めるため，図全体に対する
    # 割合と，ファイルの座標系がずれる。埋め込み用は詰めずに出す。
    import io
    buf = io.StringIO()
    fig.savefig(buf, format='svg', dpi=RASTER_DPI)
    svg = buf.getvalue()
    svg = svg[svg.index('<svg'):]          # XML 宣言と DOCTYPE を落とす

    # ---- 点の位置を図全体に対する割合で得る -----------------------------
    axes_list = list(ax) if isinstance(ax, (list, tuple, np.ndarray)) else [ax]
    W, H = fig.bbox.width, fig.bbox.height
    n_pts = len(tips)

    # **見出しは点ごとに書かない。** 点が1万個ある図では，
    # 「品詞」「頻度」…という見出しを1万回繰り返すだけで HTML が
    # 1 MB 以上ふくらむ。全点で見出しが同じなら1回だけ書き，
    # 値の並びだけを点に持たせる（JS 側で組み直す）。
    keys = [str(k) for k, _ in tips[0].get('fields', [])] if tips else []
    same_keys = bool(keys) and all(
        [str(k) for k, _ in t.get('fields', [])] == keys for t in tips)
    nheads = {str(t['notes'][0]) for t in tips if t.get('notes')}
    nhead = next(iter(nheads)) if len(nheads) == 1 else ''
    # 本文を渡さず ``notes=(見出し, None)`` としたときは，
    # ``links`` の先の語を JS 側で並べる
    linknotes = bool(nhead) and any(
        t.get('notes') and t['notes'][1] is None and t.get('links')
        for t in tips)

    pts = []
    if coords is not None and len(coords) != len(axes_list):
        raise ValueError(
            f'save_interactive: coords の数が面の数と違う'
            f'（面 {len(axes_list)} / coords {len(coords)}）')
    for k, axk in enumerate(axes_list):
      xk, yk = (coords[k] if coords is not None else (xs, ys))
      if not (len(xk) == len(yk) == n_pts):
          raise ValueError(
              f'save_interactive: 面 {k} の座標の数が情報の数と違う'
              f'（x={len(xk)}, y={len(yk)}, 情報={n_pts}）')
      pxy = axk.transData.transform(np.column_stack([np.asarray(xk, float),
                                                     np.asarray(yk, float)]))
      for j, (t, (px, py)) in enumerate(zip(tips, pxy)):
        i = k * n_pts + j
        # **変数名に注意。** ここを d と書くと，上で取った出力先 d
        # （svg_path.parent）を上書きして，最後に d / '....html' が
        # 「dict ÷ str」になる。実際に踏んだ。名前は使い回さない。
        # 添字 i は JS では使わない（行は r で引く）。点が1万個ある図では
        # 使わない値も 100 KB 単位で効くので書かない。
        rec = {'r': j, 'term': str(t.get('term', '')),
               'x': round(float(px) / W, 6),
               'y': round(1 - float(py) / H, 6)}        # SVG は上が 0
        if same_keys:
            rec['v'] = [str(b) for _, b in t.get('fields', [])]
        else:
            rec['fields'] = [[str(a), str(b)] for a, b in t.get('fields', [])]
        if t.get('notes'):
            # ('見出し', '本文') の2つ組。横に長い情報（近傍語など）。
            # 本文を None にすると，**線で結ぶ先の語を JS が並べる**
            # （同じ語の列を点ごとに書かずに済む。1万点で 1 MB 近く効く）
            if t['notes'][1] is None:
                pass
            elif nhead:
                rec['n'] = str(t['notes'][1])
            else:
                rec['notes'] = [str(t['notes'][0]), str(t['notes'][1])]
        if t.get('links'):
            # 原空間での近傍の添字。**同じ面の中で**線を結ぶ
            rec['links'] = [k * n_pts + int(q) for q in t['links']]
        pts.append(rec)

    cols = table_cols or keys
    head = f'<tr><th>{_esc(id_col)}</th>' + ''.join(
        f'<th>{_esc(c)}</th>' for c in cols) + '</tr>'
    # 数字の列だけ右寄せにする。時代名や作品 ID を右寄せにすると読みにくい。
    def _numish(v):
        t = str(v).strip().replace('%', '').replace(',', '')
        t = t.lstrip('+-')
        return bool(t) and t.replace('.', '', 1).isdigit()

    # 表に載せる点。**面の数だけ重複させない**（論理点1つに1行）
    if table_idx is None:
        order = list(range(n_pts))
    else:
        seen, order = set(), []
        for i in [int(q) for q in table_idx]:   # 重複を除きつつ順序は保つ
            if 0 <= i < n_pts and i not in seen:
                seen.add(i); order.append(i)

    # 表は **tips から作る**（点の JSON は見出しを省いてあるので）
    body = []
    for j in order:
        fv = {str(a): str(b) for a, b in tips[j].get('fields', [])}
        tds = ''
        for c in cols:
            v = fv.get(c, '')
            cls = ' class="num"' if _numish(v) else ''
            tds += f'<td{cls}>{_esc(v)}</td>'
        body.append(f'<tr data-i="{j}">'
                    f'<td>{_esc(tips[j].get("term", ""))}</td>{tds}</tr>')

    # ---- 由来を図自身に刻む -------------------------------------------
    # **どの表から描いた図かが分からないと，試験用のデータで描いた図が
    # 本物として配られる。** 実際に起きた（2026-09-22）。
    import datetime as _dt
    stamp = _dt.datetime.now().astimezone().strftime('%Y-%m-%d %H:%M')
    src = Path(source) if source else None
    prov = f'点 {len(pts)} 個／作図 {stamp}'
    warn = ''
    if src is not None:
        try:
            mt = _dt.datetime.fromtimestamp(src.stat().st_mtime).strftime('%Y-%m-%d %H:%M')
        except OSError:
            mt = '不明'
        prov = f'入力 {src.name}（更新 {mt}）／' + prov
        # プロジェクトの外（/tmp など）から描いた図は試験用である
        try:
            outside = not str(src.resolve()).startswith(str(ROOT.resolve()))
        except Exception:                               # noqa: BLE001
            outside = True
        if outside or '/tmp/' in str(src):
            warn = ('<p class="danger">⚠ <b>試験用の入力から作った図である。'
                    f'配布してはいけない。</b>（入力 {_esc(str(src))}）</p>')
            prov = f'入力 {_esc(str(src))}／' + f'点 {len(pts)} 個／作図 {stamp}'

    html = f"""<!DOCTYPE html>
<html lang="ja"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>{_esc(title or stem)}</title>
<style>{INTERACTIVE_CSS}</style></head>
<body data-nlog="{n_pts}" data-ntable="{len(body)}"><div class="wrap">
<h1>{_esc(title or stem)}</h1>
<p class="sub">{_rich(note)}</p>
{warn}
<div class="card">
  <div class="figbox">
    {svg}
    <svg id="links"></svg><svg id="marks"></svg>
    <div id="hit"></div><div id="ring"></div><div id="tip"></div>
  </div>
  <p class="hint">{_rich(hint or '点にカーソルを近づけると語が出る（最も近い点を拾うので，真上に置かなくてよい）。図の中の注記は静止版と同じものである。')}</p>
  <div class="bar">
    <input id="q" type="search" placeholder="語・作品・時代で絞り込む">
    <span id="count">{len(body)} 件</span>
    <span>·</span>
    {(f'<span>表は {len(body)} 件（図の点は {n_pts} 件。'
      '表に無い語も図の上で指せる。検索は図の印にも効く）</span><span>·</span>')
     if len(body) < n_pts else ''}
    <a href="{_esc(svg_path.name)}" download>SVG を保存</a>
    <span>（この HTML の中の図はその SVG そのもの）</span>
  </div>
  <div class="scroll"><table><thead>{head}</thead>
    <tbody>{''.join(body)}</tbody></table></div>
  <p class="prov">{prov}</p>
</div>
<script type="application/json" id="pts-data">{_json.dumps(
    {'keys': keys if same_keys else [], 'nhead': nhead,
     'linknotes': linknotes, 'pts': pts},
    ensure_ascii=False, separators=(',', ':'))}</script>
<script>{INTERACTIVE_JS}</script>
</div></body></html>
"""
    path = outdir / f'{stem}.html'
    path.write_text(html, encoding='utf-8')
    print(f'[fig] {path}  ({path.stat().st_size/1024:,.0f} KB・対話版／'
          f'点 {len(pts)} 個)')
    return svg_path, path


def _esc(s):
    """HTML の特殊文字を落とす。**語はコーパス由来なので必ず通す。**"""
    return (str(s).replace('&', '&amp;').replace('<', '&lt;')
            .replace('>', '&gt;').replace('"', '&quot;'))


def _rich(s):
    """注記の ``**…**`` だけを太字にする。

    説明文をノートブックと同じ書き方（Markdown 風）で書けるようにする。
    **先に必ず _esc を通す**ので，タグを書き込まれる余地は無い。
    ``**`` のままだと HTML では記号がそのまま出て読みにくい。
    """
    import re as _re
    return _re.sub(r'\*\*(.+?)\*\*', r'<b>\1</b>', _esc(s))


PALETTE = ['#0072B2', '#E69F00', '#009E73', '#CC79A7',
           '#56B4E9', '#D55E00', '#F0E442', '#666666']

# --- 順序のあるものを塗るための1色相のランプ -----------------------------
# **時代・年次・段階のように順序のあるものを，上の8色で塗ってはいけない。**
# 明治中期が青で明治後期が黄なら，隣り合う時代が隣り合う色にならず，
# 「時代が下るとどちらへ動くか」という肝心のことが読めなくなる。
# 1色相の濃淡にすれば，近いもの同士が近い色になり，勾配がそのまま見える。
# 散布図の点は白地の上に置くので，いちばん明るい段は 100 ではなく
# 250（背景との対比 2:1）から始める。100 は面で塗るとき（ヒートマップ）用。
SEQ_BLUE_STEPS = ['#86b6ef', '#5598e7', '#3987e5',
                  '#256abf', '#184f95', '#0d366b']
SEQ_BLUE = LinearSegmentedColormap.from_list('jlit_blue', SEQ_BLUE_STEPS)

# 離散の順序（4区分など）を塗るときはこちら。隣の段と明度差が十分あり，
# いちばん明るい段も背景から浮く（対比 2:1 以上）ことを確かめてある。
SEQ_BLUE_5 = ['#86b6ef', '#3987e5', '#256abf', '#184f95', '#0d366b']

# 大分類の2色。散布図ではどの2点も隣り合いうるので**全ペアが
# 見分けられる必要**があり，使える色数は多くない。2色に絞って，
# 下位の区別は印の形に持たせる。
GENRE_C = {'Fiction': '#2a78d6', 'Nonfiction': '#eb6834'}

# --- 初出年の5段 ---------------------------------------------------------
# 切れ目は period と同じ 1900／1912／1926／1945。
# **6段にはできない。** 1色相の濃淡で順序を見せるには，隣り合う段の明度差が
# 0.06 以上要る。この青系ランプは 250→700 で明度差にして 0.30 ほどしか幅が
# 無いので，段を6つ取るとどこかが 0.05 台に落ち，隣が見分けられなくなる。
# そこで作品数3点の明治前期（〜1886）を明治中期にまとめて5段とする。
YEAR_EDGES = [1900, 1912, 1926, 1945]
YEAR_LABELS = ['〜1899 明治前・中期', '1900-1911 明治後期', '1912-1925 大正',
               '1926-1944 昭和戦前', '1945- 昭和戦後']


def year_bands(years):
    """初出年を5段に畳み，``(段番号, ラベル, 色)`` を返す。

    段番号は 0〜4。**初出年が読めないものは -1** にする。0 に落とすと
    年の分からない作品が全部いちばん古い段に入り，通時の議論が崩れる。

    時代で塗る図はすべてこれを通すこと。同じ色が全ステップで同じ時代を
    指すようになり，Step 1 の図と Step 7 の図を並べて読める。
    """
    y = pd.to_numeric(pd.Series(list(years)), errors='coerce')
    code = np.full(len(y), -1, dtype=int)
    ok = y.notna().values
    if ok.any():
        code[ok] = np.digitize(y[ok].values, YEAR_EDGES)
    return code, YEAR_LABELS, SEQ_BLUE_5


def run_script(script, *args, tail=4000):
    """scripts/ のスクリプトを実行し，標準出力・標準エラー・終了コードを必ず表示する。

    print(r.stdout or r.stderr) では，標準出力が空でないときに
    エラーの内容が隠れてしまう。学習用には両方見えるほうがよい。
    """
    cmd = [sys.executable, str(ROOT / 'scripts' / script)] + [str(a) for a in args]
    print('$ python', ' '.join(cmd[1:]))
    r = subprocess.run(cmd, capture_output=True, text=True)

    def _tail(s):
        # 末尾 tail 文字だけを出す。行の途中で切らないよう，切ったときは
        # 次の改行から始め，前を省いたことを明示する
        if len(s) <= tail:
            return s
        s = s[-tail:]
        return '（…前略）\n' + s[s.find('\n') + 1:]

    if r.stdout:
        print(_tail(r.stdout))
    if r.stderr.strip():
        # 標準エラーには**エラー以外**も出る。MALLET は学習の進み具合
        # （<10> LL/token: …）と途中のトピック上位語をここに書く。
        # 判断は [exit 0] かどうかで行う
        print('--- stderr（進行ログを含む。エラーとは限らない）---')
        print(_tail(r.stderr))
    print(f'[exit {r.returncode}]' + ('' if r.returncode == 0 else '  ← 0 でなければ失敗'))
    return r


# 使うメタデータ。増補分（45点）を含む v3 があればそちらを優先する。
# v2 は v1 の 64 点しか無いので，増補後のコーパスで v2 を使うと
# 突合が外れて period も genre も空になる（Step 3 で v3 を作る）。
# Step 3 で自分が作った v3 は *_local.csv に書かれる（配布版は上書きしない。
# 上書きすると git pull のたびに衝突する）。自分の版 > 配布版 v3 > v2 の順。
for _m in ('corpus_metadata_v3_local.csv', 'corpus_metadata_v3.csv',
           'corpus_metadata_v2.csv'):
    META = ROOT / 'metadata' / _m
    if META.exists():
        break

# 図・表の書き出し先は自分の作業フォルダ my_work/results/。my_work/ は
# コースのリポジトリの外扱い（.gitignore）で，自分の GitHub に控えを取る。
OUT = ROOT / 'my_work' / 'results'
OUT.mkdir(parents=True, exist_ok=True)
print('OUT  =', OUT)


## 1. まず共起行列を作ってみる

word2vec に入る前に，**素朴な共起行列＋PPMI＋SVD** で同じことをやる。
Levy & Goldberg (2014) が示したとおり，両者は数学的に近い関係にある。
手で作ると，word embedding が魔法ではないことが分かる。

In [ ]:
DS  = ROOT/'data'/'datasets'
TOK = ROOT/'data'/'tokens'
docs = [f.read_text(encoding='utf-8').split()
        for f in sorted((TOK/'tokens_content').glob('*.txt'))]
print(f'{len(docs)} 文書 / {sum(len(d) for d in docs):,} 語')

WINDOW = 5
vocab = Counter(w for d in docs for w in d)
V = [w for w,c in vocab.most_common(3000)]
vi = {w:i for i,w in enumerate(V)}
C = np.zeros((len(V), len(V)), dtype=np.float32)
for d in docs:
    ids = [vi.get(w,-1) for w in d]
    for k,a in enumerate(ids):
        if a < 0: continue
        for b in ids[max(0,k-WINDOW):k+WINDOW+1]:
            if b >= 0 and b != a:
                C[a,b] += 1
print('共起行列:', C.shape, '非ゼロ率 {:.1%}'.format((C>0).mean()))

In [ ]:
# PPMI（正の相互情報量）→ SVD
tot = C.sum(); pw = C.sum(1)/tot; pc = C.sum(0)/tot
with np.errstate(divide='ignore', invalid='ignore'):
    PMI = np.log((C/tot) / (pw[:,None]*pc[None,:]))
PPMI = np.nan_to_num(np.maximum(PMI, 0))
U,S,Vt = np.linalg.svd(PPMI, full_matrices=False)
E = U[:,:100]*S[:100]
En = E/ (np.linalg.norm(E,axis=1,keepdims=True)+1e-12)

def nbr(w, k=10, M=En):
    if w not in vi: return f'{w} は語彙にない'
    s = M @ M[vi[w]]
    return ' '.join(V[i] for i in np.argsort(-s)[1:k+1])

for w in ['女','心','国','汽車','恋','自由','戦争']:
    print(f'{w:<6} → {nbr(w)}')

## 2. word2vec を学習する

In [ ]:
from gensim.models import Word2Vec
sents = [f.read_text(encoding='utf-8').split()
         for f in sorted((DS/'chunks').glob('*.txt'))] or docs
# 既定値は config/pipeline.yaml の word2vec: に揃えてある（dim 300／window 3）。
# **窓を狭くしてある**のは，語の統語的な振る舞い（品詞・共起の型）を拾いたい
# からである。窓を広げると主題の近さが優勢になる（演習 1 で確かめる）。
W2V_DIM, W2V_WIN = 300, 3
SEEDS = [11, 22, 33, 44, 55, 66, 77, 88, 99, 111]   # 報告にはこの並びをそのまま書く
w2v = Word2Vec(sents, vector_size=W2V_DIM, window=W2V_WIN, min_count=20,
               sg=1, workers=4, epochs=20, seed=SEEDS[0])
print(f'語彙: {len(w2v.wv)}'
      f'（dim={W2V_DIM} window={W2V_WIN} seed={SEEDS[0]}）')
for w in ['女','心','国','汽車','恋','自由','戦争','機械','神']:
    if w in w2v.wv:
        print(f'{w:<6} → ' + ' '.join(x for x,_ in w2v.wv.most_similar(w, topn=10)))

### 演習 1 — ハイパーパラメータの影響

`window` を 2 / 3 / 10 と変えて，同じ語の近傍がどう変わるかを見よ
（3 が本パイプラインの既定値である）。

一般に
- **小さい window（2–3）** → 統語的・構文的な類似（品詞が揃う）
- **大きい window（10–15）** → 主題的・連想的な類似（同じ話題の語）

文学の主題分析には大きめ，文体分析には小さめが向く。**既定を 3 にしてある
のは，この授業の主眼が文体にあるからである**（§4 のギャラクシーで
「品詞のまとまり」と「意味のまとまり」の両方が見えるのも，この窓幅のため）。

In [ ]:
probe = ['女','汽車','心']
# 語を行・window を列にすると，**同じ語が窓幅でどう動くか**を横に読める。
cols = {}
for win in [2, 3, 10]:
    m = Word2Vec(sents, vector_size=W2V_DIM, window=win, min_count=20,
                 sg=1, workers=4, epochs=15, seed=SEEDS[0])
    cols[f'window={win}'] = [
        ' '.join(x for x,_ in m.wv.most_similar(w, topn=8)) if w in m.wv
        else '（min_count 未満）' for w in probe]
t = pd.DataFrame(cols, index=probe)
t.index.name = '語'
show(t.reset_index(), caption='窓幅を変えると近傍語がどう変わるか（上位8語）')
print('狭い窓は統語的に置き換えられる語（品詞が同じ語）を，'
      '広い窓は主題の近い語を集めやすい。')

### 演習 2 — 安定性の検査（重要）

**乱数の種を変えると近傍語は変わる。** Antoniak & Mimno (2018) は，
小規模コーパスでは word embedding の近傍が実験ごとに大きく揺れることを示した。

種を **10 回**（`SEEDS` = 11, 22, …, 111）変えて学習し，近傍の**一致率**を
測る。一致率が低い語について「意味が変化した」と論じてはいけない。

⚠ 10 回の学習は時間がかかる（この規模で数分）。**時間が無いときは
`SEEDS[:5]` に減らしてよいが，報告には何回で測ったかを必ず書くこと。**

In [ ]:
def topn(model, w, k=10):
    return [x for x,_ in model.wv.most_similar(w, topn=k)] if w in model.wv else []

# 学習の設定は本番と同じ（dim 300／window 3）。epochs だけ 15 に落としてある。
runs = [Word2Vec(sents, vector_size=W2V_DIM, window=W2V_WIN, min_count=20,
                 sg=1, workers=4, epochs=15, seed=s) for s in SEEDS]
print(f'{len(runs)} 回学習した（種: ' + ', '.join(map(str, SEEDS)) + '）')
rows=[]
for w in ['女','心','国','汽車','恋','自由','戦争','機械','神','自然']:
    sets = [set(topn(m,w)) for m in runs]
    sets = [s for s in sets if s]
    if len(sets) < 2: continue
    jac = np.mean([len(a&b)/len(a|b)
                   for i,a in enumerate(sets) for b in sets[i+1:]])
    rows.append({'term':w, 'freq':vocab[w], 'jaccard':round(jac,3)})
stab = pd.DataFrame(rows).sort_values('jaccard')
show(stab.rename(columns={'term':'語','freq':'頻度','jaccard':'Jaccard 平均'}),
     caption=f'乱数の種を変えたときの近傍上位10語の一致'
             f'（{len(runs)}回の全ペア平均）',
     fmt={'頻度':'{:,.0f}','Jaccard 平均':'{:.3f}'})
weak = stab[stab.jaccard < 0.3]
note = ('：' + '，'.join(weak.term)) if len(weak) else ''
print(f'Jaccard が 0.3 を下回る語は {len(weak)} 語{note}。'
      '**この語の近傍は解釈に耐えない。**')
print('min_count を上げる／コーパスを増やす／複数回の平均を取る。')

In [ ]:
fig, ax = plt.subplots(figsize=(7,5))
ax.scatter(stab.freq, stab.jaccard, s=60, color=PALETTE[0], zorder=3)
ax.set_xscale('log'); ax.set_xlabel('コーパス頻度（対数）')
ax.set_ylabel(f'近傍の一致率（Jaccard, {len(runs)}試行）')
ax.axhline(.3, color=PALETTE[5], ls='--', lw=1)
ax.set_title('頻度が低い語ほど word embedding は不安定になる')
ax.spines[['top','right']].set_visible(False)
fig.tight_layout()
label_points(ax, stab.freq, stab.jaccard, stab.term, fontsize=9)
save_fig(fig, 'Step5_stability'); plt.show()

## 3. 語彙空間を眺める — **主成分分析と UMAP を並べて比べる** ★

2次元に落として語のまとまりを見る。ただし**落とし方は1つではない**。
同じ300語を**2つの方法**で落として並べる。

| | 何を守ろうとするか | 何を捨てるか |
|---|---|---|
| **主成分分析（PCA）** | **大域**の構造（分散の大きい向き）。線形 | 局所の細かい近さ。第1・2主成分に乗らない違い |
| **UMAP** | **局所**の近さ（各点の近傍） | 大域の配置・塊どうしの距離。軸の意味 |

見どころは「どちらが正しいか」ではない。**同じ空間なのに絵が違う**こと，
そして**違い方が方法の性質どおりか**である。

- PCA の第1主成分は，たいてい**頻度**か**品詞**の軸になる（語彙素の列なら
  助詞・助動詞が一方の端に寄る）。軸に解釈を与えられるのが PCA の強み。
- UMAP は塊をはっきり見せるが，**塊どうしの距離と軸には意味が無い**。
  締まった塊が出たからクラスタがあるとは言えない。

目で見るだけでは水掛け論になるので，**3つの数**を並べる。

| 指標 | 何を測るか | 強いのは |
|---|---|---|
| 信頼度（trustworthiness） | 画面で近い点が原空間でも近いか（局所） | ふつう UMAP |
| 近傍保存 | 原空間の上位10近傍のうち画面でも上位10に入る語数 | ふつう UMAP |
| 順位相関 ρ | 原空間の距離と画面の距離の順位の一致（**大域**） | ふつう PCA |

**この表が「方法を選ぶ」ということの中身である。** 図の見た目ではなく，
何を守りたいかで選び，選んだ理由を報告に書く。

In [ ]:
# ---- 同じ300語を2つの方法で落とす ------------------------------------
WORD_PROJ = 'umap'      # 'umap' / 'tsne' / 'auto'（auto のときだけ t-SNE に落ちる）
WORD_SEED = 20260920

sel = [w for w,c in vocab.most_common(400) if w in w2v.wv][:300]
X  = np.vstack([w2v.wv[w] for w in sel]).astype(np.float32)
Xn = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-12)

# (1) 主成分分析（線形・大域）。中心化してから特異値分解する
Xc = X - X.mean(0)
U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
P_pca = U[:, :2] * S[:2]
var = (S ** 2 / (S ** 2).sum())[:2]

# (2) UMAP（非線形・局所）。使えなければ**止まって理由を出す**
P_umap, METHOD_W = project(Xn, WORD_PROJ, WORD_SEED)

q_pca  = proj_quality(Xn, P_pca)
q_umap = proj_quality(Xn, P_umap)
show(pd.DataFrame([
        {'射影': f'主成分分析（第1・2主成分で {var.sum():.1%}）',
         '信頼度': q_pca['trust'], '近傍保存（/10）': q_pca['keep'].mean(),
         '順位相関 ρ（大域）': q_pca['rho']},
        {'射影': METHOD_W.split(' (')[0],
         '信頼度': q_umap['trust'], '近傍保存（/10）': q_umap['keep'].mean(),
         '順位相関 ρ（大域）': q_umap['rho']}]),
     caption=f'同じ {len(sel)} 語を2つの方法で2次元に落とした結果',
     fmt={'信頼度': '{:.3f}', '近傍保存（/10）': '{:.1f}',
          '順位相関 ρ（大域）': '{:+.3f}'})
print('局所（信頼度・近傍保存）と大域（ρ）は別物である。')
print('**どちらが上でも「良い射影」ではない。** 何を守りたいかで選ぶ。')

In [ ]:
# ---- 2面に並べて描く（静止版 SVG ＋ 対話版 HTML）----------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 7))
panels = [('主成分分析', P_pca,
           f'第1・2主成分（分散の {var.sum():.1%}）／'
           f'ρ {q_pca["rho"]:+.2f}・信頼度 {q_pca["trust"]:.2f}'),
          (METHOD_W.split(' (')[0], P_umap,
           f'{METHOD_W}／ρ {q_umap["rho"]:+.2f}・'
           f'信頼度 {q_umap["trust"]:.2f}')]
# **名前を打つ語は2面で同じにする。** 違えると見比べられない。
# 300語ぜんぶに打つと真っ黒になるので，頻度上位 40 語に絞る
# （残りの語は対話版で指せる。label_points が省いた件数を報告する）。
mark = set(sel[:40])
for ax, (name, P, sub) in zip(axes, panels):
    ax.scatter(P[:, 0], P[:, 1], s=12, color=PALETTE[0], alpha=.45,
               linewidth=0, rasterized=True)
    ax.set_title(f'{name}\n{sub}', fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)
axes[0].set_xlabel('軸に意味がある（分散の大きい向き）', fontsize=9)
axes[1].set_xlabel('※ 軸にも塊どうしの距離にも意味は無い', fontsize=9)
fig.suptitle(f'word2vec 空間（{X.shape[1]}次元・高頻度{len(sel)}語）を'
             f'2つの方法で2次元に落とす', fontsize=12)
fig.tight_layout()
# ⚠ 注記は tight_layout の**後**（軸が動くと位置がずれる）
# ⚠ 座標と名前は**同じ添字で**絞る。綴りも揃えておく（片方だけ絞る事故を
#   check_scatter_labels.py が見つけられるようにするため）
sel_arr = np.array(sel)
idx = np.array([i for i, w in enumerate(sel) if w in mark])
for ax, (name, P, sub) in zip(axes, panels):
    label_points(ax, P[idx, 0], P[idx, 1], sel_arr[idx], fontsize=7)

tips = [{'term': w,
         'fields': [('頻度', f'{vocab[w]:,}'),
                    ('近傍保存 PCA', f'{int(q_pca["keep"][i])}/10'),
                    ('近傍保存 UMAP', f'{int(q_umap["keep"][i])}/10')]}
        for i, w in enumerate(sel)]
# **面ごとに座標が違う**ので coords で渡す（渡さないと2面めの当たり判定が
# 1面めの座標で置かれ，指した点と出る語が食い違う）
save_interactive(fig, list(axes), 'Step5_wordspace',
                 P_pca[:, 0], P_pca[:, 1], tips,
                 coords=[(P_pca[:, 0], P_pca[:, 1]),
                         (P_umap[:, 0], P_umap[:, 1])],
                 source=DS/'chunks', id_col='語',
                 title='語彙空間 — 主成分分析と UMAP の比較',
                 note=(f'word2vec {X.shape[1]}次元・高頻度{len(sel)}語／'
                       f'左 主成分分析（分散 {var.sum():.1%}）／右 {METHOD_W}／'
                       f'種 {WORD_SEED}'),
                 hint=('**同じ語が左右のどこに来るか**を見る。点にカーソルを'
                       '近づけると語と，それぞれの射影での近傍保存が出る。'
                       '検索すると両方の面に印が付く。'),
                 table_cols=['頻度', '近傍保存 PCA', '近傍保存 UMAP'])
plt.show()

### 演習 — 2つの絵の違いを言葉にする

1. **同じ語**（たとえば「私」「汽車」「美しい」）が左右のどこに来るかを，
   対話版で追え。両方で端にある語，片方だけで端にある語を書き出す。
2. PCA の第1主成分の両端に来る語を10語ずつ並べ，**その軸が何か**を
   述べよ（頻度か，品詞か，語種か）。`vocab` の頻度と見比べる。
3. 上の表で ρ と信頼度がどちらに振れたかを確かめ，
   **「UMAP の絵で塊が2つに見えた」から言えること／言えないこと**を書け。
4. `WORD_SEED` を変えて UMAP だけ描き直し，塊の位置がどれだけ動くかを見よ。
   **PCA は種に依存しない**（同じ行列からは同じ答えが出る）。ここが
   「軸に意味がある」ことの実際的な意味である。

## 4. 語彙のギャラクシー — 300次元を2次元に落として眺める

近傍語の一覧は「1語ずつ」の確認である。**空間の全体像**を見たい。
300 次元は目で見られないので2次元に落とす。ここでは **UMAP** を使う
（`GAL_PROJ` で切り替える。どちらで描いたかは図と HTML に記録される）。

### ⚠ UMAP で描けているか確かめる

`GAL_PROJ = 'umap'` が既定である。**使えないときは黙って t-SNE に落ちず，
理由を出して止まる。** 図は出るのに方法だけが替わっているのが，いちばん
気づきにくい（塊の見え方が変わるので，設定の問題を分析結果と読み違える）。

入れたはずなのに使えないときの原因は，ほぼ次の4つである。

| 原因 | 見分け方 |
|---|---|
| **カーネルが作業フォルダの環境でない**（最多） | 下のセルの `sys.executable` が `dh_project/.venv/bin/python` でない |
| `umap` という**別パッケージ**が入っている | `umap.UMAP` が無いというエラー |
| numba / llvmlite が numpy の版と合わない | `import umap` 自体が例外 |
| numba がキャッシュを書けない | 初回の射影で permission のエラー |

切り分けと対処は診断スクリプトが全部やる。**ノートブックと同じ
カーネルで**走らせること。

```python
import sys, subprocess
print(sys.executable)
print(subprocess.run([sys.executable, str(ROOT/'scripts'/'check_umap.py')],
                     capture_output=True, text=True).stdout)
```

カーネルが違っていたときは，JupyterLab の右上でカーネルを
「Python (JLit)」に切り替え，**再起動する**。無ければ
`bash scripts/00_bootstrap_mac.sh`（自分の Mac は `--personal`）を実行し直す。

入っていないパッケージは，リポジトリの中で `uv add umap-learn` のように入れる。
`~/Documents/dh_project/pyproject.toml` があるので，カーネルと同じ `.venv` に入る
（**`uv sync` は使わない**）。

### この図が見せようとしていること — 文法と意味の二重構造

word embedding は「意味の似た語が近くに来る」と説明されがちだが，実際に学習して
いるのは**文脈の似た語が近くに来る**ことである。文脈が似ていれば

- **文法的にも似る**（同じ位置に立てる語＝品詞が揃う）
- **意味的にも似る**（同じ話題に出る語が集まる）

この2つが同時に起きる。そこでこの図では

- **色 = 品詞**（名詞・動詞・形容詞・副詞）… 文法的なまとまり
- **印の形 = 意味／機能のカテゴリ**（`ANCHORS`）… 意味的なまとまり

と**別の手がかりに分けて**塗ってある。見どころは，
**同じ色の中に別の形の塊がいくつもできている**ところである。
形容詞（緑）は全体として一帯に寄りながら，その中で「色の形容詞」と
「大小の形容詞」に分かれる。これが「文法的な関係と意味的な関係の両方が
モデル化されている」ということの，目に見える形である。

### ⚠ この図でいちばん大事な注意

**軸に意味は無い。** 上下左右は「第1主成分」のような解釈可能な軸ではなく，
**近さだけ**が意味を持つ。しかもその近さは
**300 次元の近さを2次元に押し込んだ結果**であって、元の近さそのものではない。

そこでこの図は，**射影がどれだけ嘘をついているか**も一緒に出す。

- `trustworthiness`（信頼度）… 2次元で近いと見える点が，原空間でも本当に
  近いか。1 に近ければ嘘が少ない
- 点ごとの**近傍保存**… その語の原空間での上位10近傍のうち，画面上でも
  上位10近傍に入っているのは何語か
- 対話版で点を指すと，**原空間での近傍10語へ線が伸びる**。
  線が遠くまで伸びる点は，その語の近傍関係が2次元に収まっていない

**線が長い語について「図で遠いから意味が遠い」と言ってはいけない。**
これがこの図の使い方であり，同時に限界の示し方である。

### 4.0 どの語の列で学習するか — 表層形・語彙素・内容語 ★

`05_tokenise_unidic.py` は**3つの列**を書き出している。どれを使うかは
好みではなく，**何を分析の対象とするか**の宣言である。

| 列 | 中身 | 何が起きるか |
|---|---|---|
| `tokens_surface` | 表層形そのまま | 「言う／言った／言ふ／云う」が**別の語**になる。異なり語数が膨らみ，1語あたりの頻度が下がる |
| `tokens_lemma` | 語彙素（見出し語）・**全品詞** | 活用と表記のゆれを畳む。助詞・助動詞・代名詞も**空間に入る** |
| `tokens_content` | 語彙素のうち名詞・動詞・形容詞・副詞 | 機能語を落とす。主題的な近さが出やすい |

**「表層形でないと空間が歪む」という直感には，半分だけ理由がある。**
分けて考える必要がある。

1. **機能語を落とすこと**（content の問題）。窓 3 で
   「彼 は 汽車 に 乗っ た」を見るとき，助詞を落とすと隣接関係が
   「彼 汽車 乗る」に変わる。つまり**窓の意味が変わり，実効的に広くなる**。
   しかも「が／を／に」対「て／た」という**品詞を見分ける最強の手がかり**
   が消える。§4 で「文法的な関係もモデル化されている」ことを見せるなら，
   機能語を落とすのは自分の首を絞めている。**この点はご指摘のとおり。**
2. **語彙素に畳むこと**（lemma の問題）。こちらは畳んだほうがよい。
   1872–1959 のコーパスでは，同じ語が旧字旧仮名・文語活用で
   **時代ごとに違う表層形**になる。表層形のままだと「時代が下ると語が
   入れ替わる」という見かけの変化が生じ，通時比較（Step 6）が壊れる。
   低頻度の異形が大量にできて，ベクトルも不安定になる。

つまり**歪むのは「表層形でないから」ではなく「機能語を落としたから」**
であり，対処は `tokens_surface` ではなく **`tokens_lemma`（全品詞・語彙素）**
である——というのが，次のセルの数字で確かめる仮説である。

⚠ 表層形が要る分析もある。**文体の指標**（旧仮名の比率・活用形の分布・
「ぬ／ず」の使い分け）は表層形でなければ測れない。`00_extend_metadata.py`
が `bungo_per10k` を測るのに `tokens_surface` を使っているのはそのためである。
**目的で選ぶ。どれかが一般に正しいのではない。**

### 凡例に埋め込む語（あらかじめ決めておく）

「どこに何があるか」の手がかりとして，カテゴリの典型語を**印の形**で
目立たせる。名詞の意味カテゴリ（身体・親族・自然・器物・抽象語）に加えて，
**動詞・形容詞・副詞のカテゴリ**を入れ，さらに `tokens_lemma` で学習した
ときは**文語の助動詞・口語の助動詞・人称代名詞**の3群が加わる。
カテゴリは `ANCHORS` に書いてあるので，**自分の問いに合わせて書き換えること**
（それがこのステップの課題でもある）。

⚠ `tokens_content` で学習したときは，**助動詞・助詞・代名詞はそもそも
空間に無い。** 「なり」「です」「私」を探しても見つからないのは，
モデルの失敗ではなく**入力の定義**である。上の3群が図から消えるので，
どちらの列で学習したかは図を見れば分かる。

⚠ 副詞の凡例語には注意が要る。「非常に」のような語は UniDic では
**「非常」＋「に」に切られる**ので，1語としては空間に無い。図の下に
「語彙に無い凡例語」として名指しされるので，そこで**辞書の切り方**を
確かめること（これも分析の一部である）。

In [ ]:
# ---- 4.0 3つの列を実測で比べる ----------------------------------------
# **結論を書く前に数える。** 小さいモデル（dim 100・10 epochs）を3つ作り，
#   * 異なり語数（表層形はどれだけ割れるか）
#   * 低頻度語の割合（割れるとベクトルが不安定になる）
#   * 機能語が語彙に入っているか
#   * 品詞の凝集度（＝文法的な構造が入っているか）
#   * 意味カテゴリの凝集度（＝意味的な構造が入っているか）
# を並べる。**本番の設定（dim 300）ではない**ので，数字の絶対値ではなく
# 列どうしの差を見ること。
CMP_RUN = True          # 時間が無いときは False（下の表は出ない）
CMP_DIM, CMP_EPOCHS, CMP_MIN = 100, 10, 20
CAP = 80000             # 1作品あたりの上限。06 の --max-chunks 40 ×
                        # --chunk 2000 と同じ量に揃える（長篇に空間を
                        # 支配させない）。**先頭から切るので，06 の
                        # 層化無作為とは抜き方が違う**（比較用と割り切る）
STREAMS = ['tokens_lemma', 'tokens_content', 'tokens_surface']

class StreamCorpus:
    # 作品ごとに読んで返す。**全体を記憶に載せない**（表層形の列は
    # 1万語どころではないので，素朴に list にすると数百 MB になる）。
    # gensim は2回以上反復するので，__iter__ で毎回読み直す。
    def __init__(self, d, cap=CAP, piece=10000):
        self.files = sorted(Path(d).glob('*.txt'))
        self.cap, self.piece = cap, piece
    def __iter__(self):
        for f in self.files:
            toks = f.read_text(encoding='utf-8').split()[:self.cap]
            for i in range(0, len(toks), self.piece):
                yield toks[i:i + self.piece]

def pos_of_stream(stream, need, limit=60000):
    # 語 → 品詞（大分類）。**表層形の列は TSV の surface 列で引く。**
    # 語彙素の列と同じ列で引くと1件も当たらない（0 埋めの事故と同型）。
    col = 1 if stream.endswith('surface') else 2
    out, need = {}, set(need)
    d = TOK/'tsv'
    if not d.exists():
        return out
    for f in sorted(d.glob('*.tsv')):
        if not need:
            break
        with open(f, encoding='utf-8-sig') as fh:
            next(fh, None); next(fh, None)
            for i, line in enumerate(fh):
                if i > limit:
                    break
                c = line.rstrip('\n').split('\t')
                if len(c) > 6 and c[col] in need:
                    out[c[col]] = c[6].split('-')[0]; need.discard(c[col])
    return out

def coh(idx, Xn):
    # 群内・群外の平均コサイン類似度を**厳密に**返す（行列を作らない）
    n, Nall = len(idx), len(Xn)
    if n < 2 or n >= Nall:
        return None
    sm = Xn[idx].sum(0); T = Xn.sum(0)
    return ((float(sm @ sm) - n) / (n * n - n),
            float(sm @ (T - sm)) / (n * (Nall - n)))

FUNC_PROBE = ('の に を は が と で も から まで '
              'だ です ます た ない れる せる たい '
              'なり けり ぬ つ たり べし ごとし き '
              '私 僕 君 彼 彼女 我 汝 あなた').split()
CMP_CATS = {   # 意味の凝集度を測る名詞3群（どの列にもある語で公平に）
    '親族': '父 母 兄 姉 弟 妹 妻 夫 娘 息子',
    '自然': '春 夏 秋 冬 雨 雪 風 花 月 山 海 空',
    '器物': '汽車 電車 写真 新聞 雑誌 時計 電報 郵便 銀行 洋服',
}
CMP_POS = ['名詞', '動詞', '形容詞', '副詞', '助動詞', '代名詞', '助詞']

cmp_rows, cmp_types = [], {}
if CMP_RUN:
    for st in STREAMS:
        d = TOK/st
        if not d.exists():
            print(f'[warn] {d} が無い（05 を --lemma-policy のまま走らせると'
                  '3つとも出る）。飛ばす。')
            continue
        corpus = StreamCorpus(d)
        cnt = Counter(w for piece in corpus for w in piece)
        m = Word2Vec(corpus, vector_size=CMP_DIM, window=W2V_WIN,
                     min_count=CMP_MIN, sg=1, workers=4,
                     epochs=CMP_EPOCHS, seed=SEEDS[0])
        V = list(m.wv.index_to_key)
        Xc = np.vstack([m.wv[w] for w in V]).astype(np.float32)
        Xc /= np.linalg.norm(Xc, axis=1, keepdims=True) + 1e-12
        idx_of = {w: i for i, w in enumerate(V)}
        pm = pos_of_stream(st, V)
        p1 = np.array([pm.get(w, '') for w in V])

        gaps = []
        for pos in CMP_POS:
            r = coh(np.where(p1 == pos)[0], Xc)
            if r:
                gaps.append(r[0] - r[1])
        sgaps = []
        for k, ws in CMP_CATS.items():
            ii = [idx_of[w] for w in ws.split() if w in idx_of]
            r = coh(np.array(ii), Xc) if len(ii) > 1 else None
            if r:
                sgaps.append(r[0] - r[1])
        low = float(np.mean([cnt[w] < 100 for w in V]))
        nf = sum(1 for w in FUNC_PROBE if w in idx_of)
        cmp_types[st] = len(V)
        cmp_rows.append({
            '語の列': st.replace('tokens_', ''),
            '延べ語数': sum(cnt.values()),
            'モデル語彙': len(V),
            '低頻度語（<100）': low,
            f'機能語（/{len(FUNC_PROBE)}）': nf,
            '品詞の凝集度（中央値）': float(np.median(gaps)) if gaps else np.nan,
            '意味の凝集度（中央値）': float(np.median(sgaps)) if sgaps else np.nan,
            '品詞を引けた語': float((p1 != '').mean())})

    if cmp_rows:
        show(pd.DataFrame(cmp_rows),
             caption=f'語の列を変えるとどうなるか'
                     f'（dim {CMP_DIM}・window {W2V_WIN}・'
                     f'min_count {CMP_MIN}・1作品 {CAP:,} 語まで）',
             fmt={'延べ語数': '{:,.0f}', 'モデル語彙': '{:,.0f}',
                  '低頻度語（<100）': '{:.1%}',
                  f'機能語（/{len(FUNC_PROBE)}）': '{:.0f}',
                  '品詞の凝集度（中央値）': '{:+.3f}',
                  '意味の凝集度（中央値）': '{:+.3f}',
                  '品詞を引けた語': '{:.0%}'})
        if 'tokens_surface' in cmp_types and 'tokens_lemma' in cmp_types:
            r = cmp_types['tokens_surface'] / max(1, cmp_types['tokens_lemma'])
            print(f'表層形の異なり語数は語彙素の {r:.2f} 倍。'
                  'これが**活用形と旧仮名で割れた分**である。')
        print('読み方: 「品詞の凝集度」が高い列は**文法的な構造**を，'
              '「意味の凝集度」が高い列は**意味的な構造**をよく写している。')
        print('機能語の数が 0 の列（content）では，'
              '**助動詞・助詞・代名詞の分析はできない**。')

In [ ]:
# ---- 4.0b ギャラクシーに使うモデルを決める ----------------------------
# 既定は **tokens_lemma**（語彙素・全品詞）。理由は上の表のとおり，
#   * 機能語が空間に入る → 文語/口語の助動詞や代名詞を図に置ける
#   * 活用と旧仮名を畳む → 低頻度の異形が減り，ベクトルが安定する
#   * 通時比較（Step 6）と同じ単位で語を数えられる
# 表層形で見たいときは 'tokens_surface' に書き換えてよい。**そのときは
# 図の副題に列名が入るので，どちらで描いたかは残る。**
GAL_STREAM = 'tokens_lemma'
if not (TOK/GAL_STREAM).exists():
    print(f'[warn] {TOK/GAL_STREAM} が無いので tokens_content で描く')
    GAL_STREAM = 'tokens_content'

# 学習し直すのは時間がかかるので，**一度作ったら保存して使い回す**。
# 設定を変えたらファイル名が変わるので，古いモデルを拾うことはない。
gal_path = OUT/f'w2v_gal_{GAL_STREAM}_d{W2V_DIM}_w{W2V_WIN}_s{SEEDS[0]}.model'
if gal_path.exists():
    GAL_W2V = Word2Vec.load(str(gal_path))
    print(f'[load] {gal_path.name}（語彙 {len(GAL_W2V.wv):,}）')
    print('       学習し直したいときはこのファイルを消すこと。')
else:
    print(f'[fit ] {GAL_STREAM} で学習する'
          f'（dim {W2V_DIM}・window {W2V_WIN}・数分かかる）')
    GAL_W2V = Word2Vec(StreamCorpus(TOK/GAL_STREAM), vector_size=W2V_DIM,
                       window=W2V_WIN, min_count=20, sg=1, workers=4,
                       epochs=20, seed=SEEDS[0])
    GAL_W2V.save(str(gal_path))
    print(f'[save] {gal_path.name}（語彙 {len(GAL_W2V.wv):,}）')

In [ ]:
# ---- 語彙のギャラクシー ----------------------------------------------
GAL_N   = 10000         # 表示する語数（モデル内の頻度上位）
GAL_K   = 10            # 近傍の数（原空間・画面ともに）
GAL_SEED = 20260920
TW_SAMPLE = 2000        # 信頼度を測る標本数（全点で測ると O(N²) になる）

# カテゴリの典型語。**印の形**で示す。名詞の意味カテゴリだけでなく，
# 動詞・形容詞・副詞のカテゴリも入れる。**形は多くても11まで**にして，
# 名前は図に直接書く（形だけで見分けさせない）。
ANCHORS = {
    '身体・知覚〈名〉':   '顔 目 手 足 胸 声 髪 唇 肩 涙 頭 腕',
    '親族・人〈名〉':     '父 母 兄 姉 弟 妹 妻 夫 娘 息子 祖母 子供',
    '自然・季節〈名〉':   '春 夏 秋 冬 雨 雪 風 花 月 山 海 空 星 桜',
    '近代の器物〈名〉':   '汽車 電車 写真 新聞 雑誌 時計 電報 郵便 銀行 洋服',
    '近代の抽象語〈名〉': '自由 権利 社会 国家 文明 科学 精神 恋愛 個人 思想 芸術',
    '感情の動詞〈動〉':   '泣く 笑う 驚く 怒る 喜ぶ 悲しむ 苦しむ 恐れる 愛する 憎む',
    '属性の形容詞〈形〉': '大きい 小さい 白い 赤い 黒い 青い 長い 短い 高い 低い 深い 美しい',
    '程度・時の副詞〈副〉': '少し やがて しばらく 突然 ふと いつも しきりに ようやく まるで すでに',
}
# **機能語の3群は，機能語が空間にある列（tokens_lemma / tokens_surface）
# でだけ足す。** tokens_content で足すと「語彙に無い凡例語」が並ぶだけで，
# 「このカテゴリは空間に無い」という誤解を生む。
ANCHORS_FUNC = {
    '文語の助動詞〈助動〉': 'なり けり ぬ つ たり べし ごとし き む らむ ず',
    '口語の助動詞〈助動〉': 'だ です ます た ない れる せる たい らしい そうだ',
    '人称代名詞〈代〉':   '私 僕 俺 君 彼 彼女 我 汝 あなた おまえ わたくし',
}
if globals().get('GAL_STREAM', 'tokens_content') != 'tokens_content':
    ANCHORS.update(ANCHORS_FUNC)
CAT_M = ['o', 's', '^', 'D', 'v', 'P', 'X', '*', '<', '>', 'h']

# 色は**品詞**に割り当てる。6色とも dataviz の検証器を全ペアで通してある
#   node scripts/validate_palette.js \
#     "#0072B2,#D55E00,#009E73,#CC79A7,#E69F00,#56B4E9" \
#     --mode light --pairs all      → ALL CHECKS PASS
# 形容詞と副詞の対は色覚差が 7.6（6–8 の下限帯）なので，**色だけに
# 頼らせない**。印の形・図に直接置くカテゴリ名・表の3つで二重に示す。
# **助詞は灰色にまとめる**（7色は散布図で見分けられない）。凡例で
# 「助詞・その他」と名指しするので，灰色の意味は曖昧にならない。
POS_C = {'名詞': '#0072B2', '動詞': '#D55E00',
         '形容詞': '#009E73', '副詞': '#CC79A7',
         '助動詞': '#E69F00', '代名詞': '#56B4E9'}
POS_GREY = '#c9c8c1'

# §4.0b で決めたモデル（既定は tokens_lemma）。無ければ §2 のモデル
wv = globals().get('GAL_W2V', w2v).wv
GAL_STREAM = globals().get('GAL_STREAM', 'tokens_content')
print(f'ギャラクシーの入力: {GAL_STREAM}（語彙 {len(wv):,}）')
def _count(w):
    try:    return int(wv.get_vecattr(w, 'count'))
    except Exception:  return int(vocab.get(w, 0))

gal = list(wv.index_to_key[:GAL_N])
acat = {}
for k, ws in ANCHORS.items():
    for w in ws.split():
        if w in wv:
            acat[w] = k
# **凡例の語は頻度で切らずに必ず入れる。** 入っていない語が凡例にあると，
# 「このカテゴリは空間に無い」と誤解される。落ちた語は下で名指しする。
extra = [w for w in acat if w not in set(gal)]
gal += extra
miss = [w for k, ws in ANCHORS.items() for w in ws.split() if w not in wv]
print(f'表示 {len(gal)} 語（頻度上位 {min(GAL_N, len(wv))} '
      f'＋ 凡例の追加 {len(extra)} 語）')
if extra:
    print('  頻度順では入らないが凡例なので追加:', ' '.join(extra))
if miss:
    print(f'  [warn] モデルの語彙に無い凡例語 {len(miss)} 語: ' + ' '.join(miss))
    print(f'         min_count 未満／{GAL_STREAM} に無い品詞'
          '／**辞書が複合語を切っている**（「非常に」→「非常」＋「に」）'
          'のいずれか。どれなのかは Step 3 の TSV で確かめられる。')
    if GAL_STREAM == 'tokens_content':
        print('         いまは内容語の列なので，**助動詞・助詞・代名詞は'
              'そもそも空間に無い**（入力の定義による）。')

X  = np.vstack([wv[w] for w in gal]).astype(np.float32)
Xn = X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-12)
print(f'原空間: {X.shape[0]} 語 × {X.shape[1]} 次元')

# ---- 2次元に落とす ----------------------------------------------------
# GAL_PROJ = 'umap'  … UMAP で描く。**使えなければ止まって理由を出す**
#            'tsne'  … t-SNE で描く
#            'auto'  … UMAP を試し，駄目なら t-SNE に落ちる（理由は出す）
#
# ⚠ 既定を 'umap' にしてある。**黙って t-SNE に落ちるのがいちばん悪い。**
# 図は出るが塊の見え方が変わるので，設定の問題を分析結果と読み違える。
# 原因の切り分けは  python3 scripts/check_umap.py  が全部やってくれる。
GAL_PROJ = 'umap'
import time as _time

# 1万語だと UMAP で1–3分，t-SNE ではもっとかかる。**待つこと。**
# 初回は numba の JIT に十数秒余分にかかる。
_t0 = _time.time()
P, METHOD = project(Xn, GAL_PROJ, GAL_SEED)
print(f'射影: {METHOD}（{_time.time()-_t0:.0f} 秒）')

# ---- 原空間の近傍と，射影がどれだけ嘘をつくか ------------------------
# ⚠ 1万語の全対行列は 10000×10000 で 400 MB になる。**作らない。**
# 行を 512 語ずつに切って上位 K だけ残す（結果は全対と同じ）。
N = len(gal)
nn_true = np.empty((N, GAL_K), dtype=np.int32)
for a in range(0, N, 512):
    b = min(a + 512, N)
    s = Xn[a:b] @ Xn.T                      # 512×N。これなら 20 MB 程度
    s[np.arange(b - a), np.arange(a, b)] = -np.inf   # 自分を外す
    idx = np.argpartition(-s, GAL_K, axis=1)[:, :GAL_K]
    ord_ = np.argsort(-np.take_along_axis(s, idx, 1), axis=1)
    nn_true[a:b] = np.take_along_axis(idx, ord_, 1)
    del s

# 画面（2次元）の近傍は k-d 木で引く。全対距離を作る必要はない
from scipy.spatial import cKDTree
_, nn_proj = cKDTree(P).query(P, k=GAL_K + 1)
nn_proj = nn_proj[:, 1:]                    # 先頭は自分自身
keep = np.array([len(set(a) & set(b)) for a, b in zip(nn_true, nn_proj)])

# 信頼度は O(N²) なので**無作為標本**で測る。標本で測ったことを明記する
from sklearn.manifold import trustworthiness
rng = np.random.default_rng(GAL_SEED)
sub = (np.arange(N) if N <= TW_SAMPLE
       else np.sort(rng.choice(N, TW_SAMPLE, replace=False)))
TW = trustworthiness(Xn[sub], P[sub], n_neighbors=GAL_K, metric='cosine')
TW_NOTE = ('全点' if len(sub) == N else f'{len(sub)} 語の無作為標本')
print(f'信頼度 trustworthiness = {TW:.3f}（{TW_NOTE}で測定）'
      f'／近傍保存 平均 {keep.mean():.1f}/{GAL_K} 語')
print('**平均で半分も残らないのが普通である。** 2次元の距離を'
      '「意味の距離」として読んではいけない。')

# ---- 品詞 -------------------------------------------------------------
# 色を品詞に使うので，ここは**図の主役**である。語彙素→品詞の対応を
# Step 3 の TSV から拾う。1万語ぶん要るので，各ファイルの先頭 60000 行
# までを見て，必要な語が揃った時点で止める。
# ⚠ **表層形の列で学習したときは TSV の surface 列で引く。**
# 語彙素の列で引くと1件も当たらず，全部が灰色になる（0 埋めの事故と同型）。
POS_COL = 1 if GAL_STREAM.endswith('surface') else 2
posmap, need = {}, set(gal)
tsvdir = TOK/'tsv'
if tsvdir.exists():
    for f in sorted(tsvdir.glob('*.tsv')):
        if not need: break
        with open(f, encoding='utf-8-sig') as fh:
            next(fh, None); next(fh, None)
            for i, line in enumerate(fh):
                if i > 60000: break
                c = line.rstrip('\n').split('\t')
                if len(c) > 6 and c[POS_COL] in need:
                    posmap[c[POS_COL]] = c[6]; need.discard(c[POS_COL])
else:
    print('[info ] data/tokens/tsv が無い')

# 残った語は1語ずつ辞書に当てる（本文の文脈は無いので推定になる）。
# **文脈なしの解析なので，多義の語では取りこぼす。** それでも
# 色が塗れないより良い。何語をこの方法で埋めたかは下に出す。
n_iso = 0
if need:
    try:
        import fugashi, yaml
        _dd = (yaml.safe_load(open(ROOT/'config'/'pipeline.yaml',
                                   encoding='utf-8'))
               .get('tokenise', {}).get('dicdir'))
        tg = fugashi.GenericTagger(f'-d {_dd}') if _dd else fugashi.Tagger()
        for w in list(need):
            ws = tg(w)
            if len(ws) == 1:
                p = str(ws[0].feature[0])
                if p:
                    posmap[w] = p; need.discard(w); n_iso += 1
    except Exception as e:                            # noqa: BLE001
        print(f'[info ] 単語単位の品詞付与は使えない（{type(e).__name__}）')

pos1 = np.array([str(posmap.get(w, '')).split('-')[0] for w in gal])
cov = float(np.isin(pos1, list(POS_C)).mean())
print(f'品詞を引けた語: {len(posmap)}/{len(gal)}'
      + (f'（うち {n_iso} 語は単語単位で推定）' if n_iso else '')
      + f'／4品詞に収まった語 {cov:.1%}')
# **覆えていないのに色を塗ると，灰色の意味が「不明」から「その他の品詞」へ
# 静かにすり替わる。** 6割を下回るときは色分けをやめ，そのことを図に書く。
POS_OK = cov >= 0.60
if not POS_OK:
    print('[warn] 品詞の取得率が低いので**色分けはしない**。'
          'Step 3 を走らせて data/tokens/tsv を作ると色が付く。')

cat = np.array([acat.get(w, '') for w in gal])
print(f'凡例のカテゴリに入る語: {int((cat != "").sum())} 語／'
      f'その他 {int((cat == "").sum())} 語')


In [ ]:
# ---- 図（静止版 SVG ＋ 対話版 HTML）----------------------------------
# **色 = 品詞（文法）／印の形 = カテゴリ（意味）** の二重の塗り分け。
# 凡例も2つ出す（ax.add_artist で重ねる）。
fig, ax = plt.subplots(figsize=(13.5, 9.6))
other = cat == ''

# 地の1万語。**品詞ごとに分けて描く**ことで，凡例の色が実体と結びつく。
# 1万点はラスタ化しないと SVG が数十 MB になる
if POS_OK:
    for p, c in POS_C.items():
        mk = other & (pos1 == p)
        if mk.any():
            ax.scatter(P[mk, 0], P[mk, 1], s=4.5, color=c, alpha=.40,
                       linewidth=0, rasterized=True)
    mk = other & ~np.isin(pos1, list(POS_C))
    if mk.any():
        ax.scatter(P[mk, 0], P[mk, 1], s=4.5, color=POS_GREY, alpha=.45,
                   linewidth=0, rasterized=True)
else:
    ax.scatter(P[other, 0], P[other, 1], s=4.5, color=POS_GREY, alpha=.5,
               linewidth=0, rasterized=True)

# 凡例の語。形はカテゴリ，中の色は品詞（＝その語が実際に何と解析されたか）
cx, cy, cl = [], [], []
for ci, k in enumerate(ANCHORS):
    mk = cat == k
    if not mk.any():
        continue
    fc = [POS_C.get(p, '#6f6d66') if POS_OK else '#33322e' for p in pos1[mk]]
    ax.scatter(P[mk, 0], P[mk, 1], s=(100 if CAT_M[ci] == '*' else 62),
               c=fc, marker=CAT_M[ci], edgecolor='white', linewidth=.8,
               zorder=3)
    # 位置は平均ではなく中央値。外れた1語で札の位置が飛ばないように。
    cx.append(float(np.median(P[mk, 0]))); cy.append(float(np.median(P[mk, 1])))
    cl.append(k)

ax.set_title(
    f'語彙のギャラクシー（{GAL_STREAM.replace("tokens_", "")}・'
    f'word2vec {X.shape[1]}次元 → {METHOD.split()[0]} 2次元）'
    f'／{len(gal):,} 語・信頼度 {TW:.2f}・近傍保存 {keep.mean():.1f}/{GAL_K}')
ax.set_xlabel('※ 軸に意味は無い。近さだけが意味を持つ（しかも近さも射影の結果である）'
              '／色＝品詞（文法的なまとまり）・印の形＝カテゴリ（意味的なまとまり）')
ax.set_xticks([]); ax.set_yticks([])
for sp in ax.spines.values():
    sp.set_visible(False)

from matplotlib.lines import Line2D
h_pos = []
if POS_OK:
    for p, c in POS_C.items():
        n = int((pos1 == p).sum())
        h_pos.append(Line2D([], [], marker='o', ls='', ms=7, color=c,
                            label=f'{p}（{n:,}）'))
    n = int((~np.isin(pos1, list(POS_C))).sum())
    if n:
        # **灰色が何なのかを名指しする。** 「その他」だけだと，
        # 引けなかった語と助詞の区別がつかない。
        n_jo = int((pos1 == '助詞').sum())
        lab = (f'助詞（{n_jo:,}）・その他・不明（{n - n_jo:,}）'
               if n_jo else f'その他・不明（{n:,}）')
        h_pos.append(Line2D([], [], marker='o', ls='', ms=7, color=POS_GREY,
                            label=lab))
h_cat = [Line2D([], [], marker=CAT_M[ci], ls='', ms=(11 if CAT_M[ci] == '*' else 8),
                color='#55534d', markeredgecolor='white',
                label=f'{k}（{int((cat == k).sum())}）')
         for ci, k in enumerate(ANCHORS) if (cat == k).any()]
if h_pos:
    lg1 = ax.legend(handles=h_pos, title='色 ＝ 品詞', frameon=False,
                    fontsize=8.5, title_fontsize=9, loc='upper left',
                    bbox_to_anchor=(1.01, 1.0), borderaxespad=0)
    ax.add_artist(lg1)
ax.legend(handles=h_cat, title='印の形 ＝ カテゴリ', frameon=False,
          fontsize=8.5, title_fontsize=9, loc='upper left',
          bbox_to_anchor=(1.01, 0.70 if h_pos else 1.0), borderaxespad=0)
fig.tight_layout()
# 凡例2つを面の外に置いてあるので，**右の余白を確保する**。
# しないと HTML に埋め込む版（切り取らない版）で凡例が切れる。
reserve_right(fig, 0.78)
# カテゴリ名は図に直接置く（色と形だけに頼らせない）。
# ⚠ label_points は tight_layout の**後**に呼ぶこと（座標が動く）
label_points(ax, cx, cy, cl, fontsize=10, color='#33322e')

tips = []
for i, w in enumerate(gal):
    tips.append({
        'term': w,
        'fields': [('品詞', posmap.get(w, '—')),
                   ('頻度', f'{_count(w):,}'),
                   ('カテゴリ', acat.get(w, '（その他）')),
                   ('近傍保存', f'{int(keep[i])}/{GAL_K}')],
        # 原空間での近傍。**画面の近さではない**ので必ず併記する。
        # 本文は None にして links から並べさせる（HTML が 1 MB 近く軽くなる）
        'notes': ('原空間の近傍:', None),
        'links': [int(j) for j in nn_true[i]],
    })

# 表は全1万件を載せると HTML が数 MB になり，読む側にも役に立たない。
# **凡例の語を先に置き**，あとは頻度上位から埋める（点はすべて指せる）。
TABLE_MAX = 600
t_idx = [i for i, w in enumerate(gal) if w in acat]
t_idx += [i for i in range(len(gal)) if i not in set(t_idx)][:max(
    0, TABLE_MAX - len(t_idx))]

save_interactive(fig, ax, 'Step5_galaxy', P[:, 0], P[:, 1], tips,
                 source=DS/'chunks', id_col='語',
                 title=f'語彙のギャラクシー（{METHOD}）',
                 note=(f'語の列 {GAL_STREAM}／'
                       f'word2vec {X.shape[1]}次元（window {W2V_WIN}）を'
                       f'2次元に射影／{len(gal):,} 語／'
                       f'信頼度 {TW:.3f}（{TW_NOTE}）／'
                       f'近傍保存 平均 {keep.mean():.1f}/{GAL_K}／'
                       f'種 {GAL_SEED}／色＝品詞・印の形＝カテゴリ'),
                 hint=('点にカーソルを近づけると語が出て，**原空間での近傍10語'
                       'へ線が伸びる**。線が遠くまで伸びる点は，その語の近傍が'
                       '2次元に収まっていない。検索すると図の上にも印が付く'
                       '（表に無い語も図の上では指せる）。'),
                 table_cols=['品詞', '頻度', 'カテゴリ', '近傍保存'],
                 table_idx=t_idx)
plt.show()

### まとまっているのは本当か — 目で見ずに数える

「親族語が固まって見える」のは，**こちらがそう並べたから**かもしれない。
射影の癖でそう見えているだけかもしれない。原空間（300次元）で測り直す。
群内の平均コサイン類似度が，群外との平均よりどれだけ高いかを見る。

⚠ 1万語の全対類似度行列は作らない（400 MB になる）。平均は
和のノルムから**厳密に**出せる。群の和を s，群の語数を n とすると

    Σ_{i,j∈群} x_i·x_j = ‖s‖²  ⇒  群内の平均 = (‖s‖² − n) / (n² − n)

（対角の n 個は自分自身との類似度 1 なので引く。全対を並べるのと同じ値で，
記憶は使わない。）まず**品詞**（文法），次に**カテゴリ**（意味）で測る。

In [ ]:
def coh(idx, Xn):
    # 群内・群外の平均コサイン類似度を**厳密に**返す（行列を作らない）
    n, Nall = len(idx), len(Xn)
    if n < 2 or n >= Nall:
        return None
    s = Xn[idx].sum(0)
    T = Xn.sum(0)
    intra = (float(s @ s) - n) / (n * n - n)
    inter = float(s @ (T - s)) / (n * (Nall - n))
    return intra, inter

T_all = Xn.sum(0)
allmean = (float(T_all @ T_all) - len(Xn)) / (len(Xn) ** 2 - len(Xn))

rows = []
if POS_OK:
    for p in POS_C:
        idx = np.where(pos1 == p)[0]
        r = coh(idx, Xn)
        if r is None: continue
        rows.append({'品詞': p, '語数': len(idx), '品詞内': r[0],
                     '品詞外': r[1], '差': r[0] - r[1],
                     '近傍保存の中央値': float(np.median(keep[idx]))})
    show(pd.DataFrame(rows).sort_values('差', ascending=False),
         caption=f'**文法的なまとまり**：品詞の凝集度'
                 f'（原空間のコサイン類似度／全体平均 {allmean:+.3f}）',
         fmt={'品詞内': '{:+.3f}', '品詞外': '{:+.3f}', '差': '{:+.3f}',
              '語数': '{:,.0f}', '近傍保存の中央値': '{:.1f}'})
    print('差が正なら，**同じ品詞の語は互いに近い**。')
    print('window を 3 に狭めてあるので，この差は大きく出るはずである'
          '（窓を 10 に広げて測り直すと縮む。演習 1 の続きとしてやってみよ）。')
else:
    print('[info ] 品詞が引けていないので品詞の凝集度は測れない。')

In [ ]:
rows = []
for k in ANCHORS:
    idx = np.where(cat == k)[0]
    r = coh(idx, Xn)
    if r is None:
        continue
    # 画面上での散らばり（中央値からの距離の中央値）も出す
    spread = float(np.median(np.hypot(P[idx, 0] - np.median(P[idx, 0]),
                                      P[idx, 1] - np.median(P[idx, 1]))))
    rows.append({'カテゴリ': k, '語数': len(idx),
                 'カテゴリ内の平均類似度': r[0],
                 'カテゴリ外との平均類似度': r[1],
                 '差': r[0] - r[1],
                 '画面上の散らばり': spread,
                 '近傍保存の中央値': float(np.median(keep[idx]))})
show(pd.DataFrame(rows).sort_values('差', ascending=False),
     caption=f'**意味的なまとまり**：カテゴリの凝集度'
             f'（原空間のコサイン類似度／全体平均 {allmean:+.3f}）',
     fmt={'カテゴリ内の平均類似度': '{:+.3f}', 'カテゴリ外との平均類似度': '{:+.3f}',
          '差': '{:+.3f}', '画面上の散らばり': '{:.2f}',
          '近傍保存の中央値': '{:.1f}'})
print('「差」が大きいカテゴリは，**空間の中で実際にまとまっている**。')
print('小さいカテゴリは，こちらが名前でまとめただけで，モデルはそう見ていない。')
print('画面上の散らばりが小さいのに差も小さい場合は，**射影がまとめて見せている**'
      'だけの可能性がある。原空間の数字のほうを信じること。')
print()
print('**2つの表を並べて読むこと。** 品詞でも差が出て，同じ品詞の中の'
      '意味カテゴリでも差が出る——これが「文法的な関係と意味的な関係の'
      '両方がモデル化されている」ということの中身である。')

### 演習 3 — ギャラクシーを自分の問いで塗り替える

1. `ANCHORS` を**自分の問いのカテゴリ**に書き換えよ（8つまで＝印の形の数）。
   例: 視覚語／聴覚語・和語／漢語・自然主義の語彙／プロレタリア文学の語彙。
   **品詞をまたぐカテゴリ**（例「戦争を語る語」＝名詞＋動詞＋形容詞）を
   立てると，色がばらけて形だけが揃う。これは
   「意味では揃うが文法では揃わない」カテゴリの見え方である。
   書き換えたら**凝集度の表**を見て，そのカテゴリが空間でまとまっているかを
   確かめる。まとまっていなければ，カテゴリの立て方を疑う。
2. **近傍保存が低い語**（対話版で線が遠くまで伸びる語）を3語選び，
   なぜ2次元に収まらないのかを考えよ。多義語か，頻度が低いか，
   複数の文脈にまたがる語か。
3. `GAL_N` を 1000 / 3000 / 10000 と変え，**信頼度がどう変わるか**を記録せよ。
   語を増やすと図は賑やかになるが，射影の嘘は増えるか減るか。
   （`W2V_WIN` を 10 にして学習し直し，**品詞の凝集度がどう動くか**も見よ。
   狭い窓が文法を拾っているという説明が正しければ，差は縮むはずである。）
4. `min_dist` を 0.0 / 0.12 / 0.5 と変えると，塊の締まり方が変わる。
   **これは分析結果ではなく作図の設定である。** 締まった図を見て
   「クラスタがある」と言ってよいか，理由とともに述べよ。
5. ⚠ UMAP / t-SNE の座標は**種に依存する**。`GAL_SEED` を変えて2回描き，
   同じ結論が言えるかを確かめよ。言えないなら，その結論は図の癖である。

## 5. このステップの課題

**提出先**：Zulip（dh-uosaka.zulipchat.com）の非公開チャネル **2026年度テクスト分析論B** ＞ トピック **Step 5**。
本文はメッセージにそのまま書き（Markdown が使える），図（SVG）と表は添付する。再提出は元の投稿を直さず，同じトピックに新しく投稿する（手順書 §5.3）。

1. 共起行列＋PPMI＋SVD と word2vec の近傍を5語について比較し，
   違いを記述すること。**どちらが良いかではなく，何が違うか**を書く。
2. `window` と `min_count` を変えた3条件で，同じ語の近傍表を作ること。
3. 安定性検査を自分の関心のある語10語で行い，
   **解釈に耐える語／耐えない語**を分類すること。
4. 次の Step のために，自分が通時的に追いたい語を **5語**選び，
   選んだ理由（文学史的な仮説）を書いてくること。

### このステップの到達点（次へ進む条件）

- コーパス全体で word2vec を学習でき，近傍語を取り出せる
- 種を変えた複数回の学習で，近傍の一致率（Jaccard）を測れる
- **解釈に耐える語と耐えない語**を区別でき，その基準を言える
- 通時的に追う5語を選び，仮説を文章にしてある
